## Notebook for general controls which are within the design fasta and the variant map and region bed
- check one at a time, because of the different styles and possible errors 

In [1]:
from importlib import reload
import pandas as pd
import sys
import os
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [2]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source,
                       col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]


# function to add a list of SPDI values from the REF to the metadata table
def add_spdi_values_2_reference(row, ref_alt_dict, alt_spdi_dict, alt_variant_pos_dict, alt_variant_class_dict):
    if hf.is_reference(row[col_allele]):
        # check if row[col_name] is in ref_alt_dict
        if not row[col_name] in ref_alt_dict:
            # raise exception
            raise ValueError('Reference ID not found in ref_alt_dict')
        row[col_SPDI] = [alt_spdi_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        # NOTE: within variant_pos: for reference sequences a array of variant positions need to be added
        row[col_variant_pos] = [int(alt_variant_pos_dict[alt_id]) for alt_id in ref_alt_dict[row[col_name]]]
        row[col_variant_class] = [alt_variant_class_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        row[col_allele] = ['ref' for _ in ref_alt_dict[row[col_name]]]
    return row

# function to generate arrays out off the columns
def make_column_arrays(row):
    """
    create arrays for the required columns
    """
    allele = row[col_allele]
    SPDI = row[col_SPDI]
    variant_pos = row[col_variant_pos]
    variant_class = row[col_variant_class]

    if allele == 'ref' or allele == 'alt': # only "alt" is string
        row[col_allele] = [allele]
    if isinstance(SPDI, str): # only for alt sequences this is true
        if SPDI != "NA":
            row[col_SPDI] = [SPDI]
    if isinstance(variant_pos, float):
        row[col_variant_pos] = [int(variant_pos)]
    elif isinstance(variant_pos, int):
        row[col_variant_pos] = [int(variant_pos)]
    if isinstance(variant_class, str):
        if row[col_variant_class] in ['SNV', 'indel']:
                row[col_variant_class] = [variant_class]
    if not isinstance(row[col_class], str):
        print(row[col_name])
    return row

In [3]:
import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
# split the metadata file headers by '#'
# Apply the function to each row and concatenate the results
pre_metadata_df_split = pd.concat(pre_metadata_df.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# Reset the index
pre_metadata_df_split.reset_index(drop=True, inplace=True)

pre_metadata_df = pre_metadata_df_split.copy()
print(pre_metadata_df.shape[0]) # 80804
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))

# variant map
variant_map_path = config['variant_region_map']
variant_map = pd.read_csv(variant_map_path, sep="\t")
# variant_map.columns = ['ID', 'Region', 'REF', 'ALT']
# variant_map.to_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map_new_colnames.tsv.gz', sep="\t", compression='gzip', index=False)
variant_map['tmp_label'] = variant_map['ID'].apply(hf.get_label)

# region bed
region_bed = config['region_bed']
region_bed = pd.read_csv(region_bed, sep="\t", header=None)
region_bed.columns = [f'region_{col_name}' for col_name in ['chr', 'start', 'end', 'name', 'score', 'strand']]

# vcf
vcf_path = config['variant_vcf']
vcf_df = pd.read_csv(vcf_path, sep='\t', comment="#", header=None)
vcf_df.columns = ['CHROM', 'var_pos', 'ID', 'vcf_REF', 'vcf_ALT', 'QUAL', 'FILTER', 'INFO']


80803


In [4]:
pre_metadata_df.head()

,name,sequence,tmp_label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random


In [5]:
# check for duplicates in pre_metadata_df (same header and same sequence) => no duplicated headers and sequences
pre_metadata_df.duplicated(subset=['name', 'sequence'],keep=False).sum()

0

In [6]:
variant_map['tmp_label'].value_counts()

tmp_label
cardiac_neuro_cava_random    46374
GC_Selvarajan                  198
GC_Kircher                     198
GC_Mendelian_variants          174
C_positive_heart_CAD            49
GC_Atrial_fib                   23
GC_Mohlke                       20
GC_Liang                         8
Name: count, dtype: int64

In [7]:
variant_region_map = variant_map.merge(region_bed, left_on='Region', right_on='region_name', how='inner')
variant_region_map

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2191262,2191532,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2191971,2192241,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2192249,2192519,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2192936,2193206,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
...,...,...,...,...,...,...,...,...,...,...,...
46816,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274993,109275263,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+
46817,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109275105,109275375,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+
46818,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274993,109275263,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+
46819,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109275105,109275375,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+


In [8]:
# pre_metadata_df['tmp_label'].value_counts()

#### Focus on missing groups: GC_Mendelian_variants, C_positive_heart_CAD
- variant region map
  - find the variant id and get the variant position => is the variant in the middle of the region
    - Mendelian variants: not in the region.bed

In [9]:
mendelian_variant_map_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/final_design/results/final_design/GC_Mendelian_variants/variant_region_map.tsv.gz'
mendelian_variant_map_path = config['mendelian_variant_map']
mendelian_variant_map = pd.read_csv(mendelian_variant_map_path, sep="\t")
mendelian_variant_map.columns = ['ID', 'Region', 'REF', 'ALT']

# GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4
mendelian_variant_map

,ID,Region,REF,ALT
0,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,GC_Mendelian_variants:ALT_chr1:21564170G>A|ALP...
1,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:REF_chr1:209816133C>CA|IRF6,GC_Mendelian_variants:ALT_chr1:209816133C>CA|I...
2,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A,GC_Mendelian_variants:ALT_chr10:23219376A>C|PT...
3,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219434A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219434A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219434A>G|PT...
4,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219436A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219436A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219436A>G|PT...
...,...,...,...,...
169,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:REF_chr9:101435912C>T|ALDOB,GC_Mendelian_variants:ALT_chr9:101435912C>T|AL...
170,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:REF_chrX:38352331A>G|OTC,GC_Mendelian_variants:ALT_chrX:38352331A>G|OTC...
171,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:REF_chrX:55028202A>G|ALAS2,GC_Mendelian_variants:ALT_chrX:55028202A>G|ALA...
172,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:REF_chrX:55031184G>C|ALAS2,GC_Mendelian_variants:ALT_chrX:55031184G>C|ALA...


In [10]:
mendelian_variant_map.sample(5)

,ID,Region,REF,ALT
92,GC_Mendelian_variants:chr7:156791474G>A|SHH,GC_Mendelian_variants:chr7:156791474G>A|SHH,GC_Mendelian_variants:REF_chr7:156791474G>A|SHH,GC_Mendelian_variants:ALT_chr7:156791474G>A|SH...
111,GC_Mendelian_variants:chr7:156791542A>C|SHH,GC_Mendelian_variants:chr7:156791472C>G|SHH,GC_Mendelian_variants:REF_chr7:156791472C>G|SHH,GC_Mendelian_variants:ALT_chr7:156791472C>G|SH...
26,GC_Mendelian_variants:chr11:5254895G>T|HBG2,GC_Mendelian_variants:chr11:5254956A>G|HBG2,GC_Mendelian_variants:REF_chr11:5254956A>G|HBG2,GC_Mendelian_variants:ALT_chr11:5254956A>G|HBG...
156,GC_Mendelian_variants:chr7:156791581A>G|SHH,GC_Mendelian_variants:chr7:156791581A>G|SHH,GC_Mendelian_variants:REF_chr7:156791581A>G|SHH,GC_Mendelian_variants:ALT_chr7:156791581A>G|SH...
64,GC_Mendelian_variants:chr7:156791459T>C|SHH,GC_Mendelian_variants:chr7:156791542A>C|SHH,GC_Mendelian_variants:REF_chr7:156791542A>C|SHH,GC_Mendelian_variants:ALT_chr7:156791542A>C|SH...


In [11]:
region_bed.loc[region_bed['region_name'].str.contains('GC_Mendelian_variants')]
region_bed.loc[region_bed['region_name'].str.contains('C_positive_heart_CAD')]

,region_chr,region_start,region_end,region_name,region_score,region_strand


In [12]:
variant_region_map['tmp_label'].value_counts()

tmp_label
cardiac_neuro_cava_random    46374
GC_Selvarajan                  198
GC_Kircher                     198
GC_Atrial_fib                   23
GC_Mohlke                       20
GC_Liang                         8
Name: count, dtype: int64

In [13]:
variant_region_map.loc[variant_region_map['ALT'].str.contains('Mendelian_variants')]

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand


In [14]:
example_mendelian_variants = mendelian_variant_map[mendelian_variant_map['REF'] == 'GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A'].ALT.to_list()
example_mendelian_variants
pre_metadata_df.loc[pre_metadata_df[col_name] == mendelian_variant_map['REF'].to_list()[0]]

,name,sequence,tmp_label
74608,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,AGGACCGGATCAACTCCCCAGGGAAATCTGTGGGCATTGTGACCAC...,GC_Mendelian_variants


#### Focus on metadata from bed and variant map: 'cardiac_neuro_cava_random', 'GC_Selvarajan', 'GC_Kircher', 'GC_Atrial_fib', 'GC_Mohlke', 'GC_Liang'

In [203]:
variant_region_map_group_names = variant_region_map['tmp_label'].value_counts().reset_index()['tmp_label'].to_list()
# cardiac_neuro_cava_random    46374
# GC_Selvarajan                  198
# GC_Kircher                     198
# GC_Mendelian_variants          174
# C_positive_heart_CAD            49
# GC_Atrial_fib                   23
# GC_Mohlke                       20
# GC_Liang                         8

In [204]:
variant_groups = variant_map['tmp_label'].value_counts().reset_index()['tmp_label'].to_list()

In [205]:
pre_metadata_df['tmp_label'].value_counts()

tmp_label
cardiac_neuro_cava_random            73940
MK                                    2906
C_positive_heart_AB                    909
GC_Selvarajan                          364
GC_Vista                               256
C_negative_heart_MK                    243
C_negative_neuron_MK                   234
GC_Mendelian_variants                  221
C_negative_neuron_NP                   217
GC_Kircher                             203
C_SLEA                                 200
GC_Cort_Chengyu                        185
C_positive_neuron_MK                   100
C_positive_heart_CAD                    99
C_positive_neuron_NP                    99
C_positive_heart_MK                     97
C_positive_neuron_CD                    96
GC_GABA_Chengyu                         85
GC_Glut_Chengyu                         83
GC_DNase_positive_shuffeled             55
GC_Atrial_fib                           45
GC_DNase_positive                       41
GC_Mohlke                               38
G

## GC_Liang (n: 16)
- 8 variants

In [ ]:
def is_variant_related(name, variant_related_list):
    return name in variant_related_list

def is_reference_related(name, reference_related_list):
    return name in reference_related_list

def is_alternative_related(name, variant_related_list):
    return name in variant_related_list


# Combine columns
def combine_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith("_x"):
            base_name = col[:-2]  # Remove "_x"
            corresponding_y = base_name + "_y"
            # Combine _x and _y columns
            if corresponding_y in df.columns:
                new_columns[base_name] = df[col].fillna(df[corresponding_y])
            else:
                new_columns[base_name] = df[col]
        elif not col.endswith("_y"):  # Add columns that aren't paired with _x/_y
            new_columns[col] = df[col]
    return pd.DataFrame(new_columns)


def get_var_pos(row, var_pos_column, allele_column, seq_start_column, variant_1_based=True, start_0_based=True, sequence_length=270):
    """
    Computes the variant pos (0-based coordinate of variant in the string)
    for the alternative sequences with different cased of the data
    (variant_1_based and start_0_based is the default)

    If variant is 0 based set variant_1_based to false (same goes for start)
    """
    variant_position = pd.NA
    if hf.is_alternative(row[allele_column]):
        variant_0_based = row[var_pos_column] - 1 if variant_1_based else row[var_pos_column]
        seq_start_0_based = row[seq_start_column] if start_0_based else row[seq_start_column] - 1
        variant_position = variant_0_based - seq_start_0_based
        if row[col_strand] == '-':
            variant_position = sequence_length - (variant_position + 1) # 0-based variant position
    return variant_position

import re
def find_indel_pattern(row, ref_column, alt_column):
    """Check if in ref or alt is more than 1 subsequent nucleotide indicating an indel
    Special case: I want it to check for the notation of muliallelic variants as well (A,T) if any of these is an indel"""
    if hf.is_alternative(row[col_allele]):
        pattern = r'(^[ACGT]{2,})|(,[ACGT]{2,})'
        # Check if the text matches the pattern
        ref_indel = bool(re.match(pattern, row[ref_column]))
        alt_indel = bool(re.match(pattern, row[alt_column]))
        return ref_indel or alt_indel
    else: False



# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

In [207]:
group_name = 'GC_Liang'
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
expected_number = pre_metadata_df_group.shape[0]
variant_region_df_group = variant_region_map.loc[variant_region_map['tmp_label'] == group_name].copy()
region_bed_group = region_bed.loc[region_bed['region_name'].str.contains(group_name)].copy()
vcf_df_group = vcf_df.loc[vcf_df['ID'].str.contains(group_name)].copy()

In [208]:
print('Expected number: ',expected_number)

Expected number:  16


In [209]:
# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])

# if variant related or element
variant_related_list = set(variant_region_df_group['REF'].to_list()).union(set(variant_region_df_group['ALT'].to_list()))
pre_metadata_df_group[col_category] = pre_metadata_df_group[col_name].apply(lambda name: 'variant' if is_variant_related(name, variant_related_list) else 'element')


pre_metadata_df_group[col_class] = pre_metadata_df_group[col_name].apply(lambda name: 'variant negative control' if is_variant_related(name, variant_related_list) else 'element inactive control')
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add allele
alternative_related_list = set(variant_region_df_group['ALT'].to_list())
reference_related_list = set(variant_region_df_group['REF'].to_list())
pre_metadata_df_group[col_allele] = pre_metadata_df_group[col_name].apply(lambda name: 'alt' if is_alternative_related(name, alternative_related_list) else 'ref' if is_reference_related(name, reference_related_list) else 'NA')

pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

### Focus on Elements

In [210]:
pre_metadata_df_group_element = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'element']
pre_metadata_df_group_element.name.to_list()

[]

### Focus on Variants

In [211]:
pre_metadata_df_group_variant = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'variant'].copy()
print(pre_metadata_df_group_variant.shape[0])
pre_metadata_df_group_variant[col_name].nunique()

16


16

In [212]:
print(f'Expected variant number of this group: {variant_region_df_group.shape[0]}')

Expected variant number of this group: 8


#### Match variant information from vcf file

In [213]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group, on='ID', how='inner')

In [214]:
variant_region_vcf_group.head()

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand,CHROM,var_pos,vcf_REF,vcf_ALT,QUAL,FILTER,INFO
0,GC_Liang:rs2125358,GC_Liang:rs2125358|Liang_fwd_tile1-1,GC_Liang:REF_rs2125358|Liang_fwd_tile1-1,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,GC_Liang,chr11,86515771,86516041,GC_Liang:rs2125358|Liang_fwd_tile1-1,.,+,chr11,86515916,G,C,.,PASS,rsid=rs2125358;Region=rs2125358|Liang_fwd_tile...
1,GC_Liang:rs17882077,GC_Liang:rs17882077|Liang_fwd_tile1-1,GC_Liang:REF_rs17882077|Liang_fwd_tile1-1,GC_Liang:ALT_rs17882077|Liang_fwd_tile1-1_rs17...,GC_Liang,chr14,22848437,22848707,GC_Liang:rs17882077|Liang_fwd_tile1-1,.,+,chr14,22848616,A,G,.,PASS,rsid=rs17882077;Region=rs17882077|Liang_fwd_ti...
2,GC_Liang:rs10502466,GC_Liang:rs10502466|Liang_fwd_tile1-1,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10...,GC_Liang,chr18,25019523,25019793,GC_Liang:rs10502466|Liang_fwd_tile1-1,.,+,chr18,25019730,G,A,.,PASS,rsid=rs10502466;Region=rs10502466|Liang_fwd_ti...
3,GC_Liang:rs1036014,GC_Liang:rs1036014|Liang_fwd_tile1-1,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...,GC_Liang,chr2,208556887,208557157,GC_Liang:rs1036014|Liang_fwd_tile1-1,.,+,chr2,208557023,C,G,.,PASS,rsid=rs1036014;Region=rs1036014|Liang_fwd_tile...
4,GC_Liang:rs2838227,GC_Liang:rs2838227|Liang_fwd_tile1-1,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs283...,GC_Liang,chr21,43270840,43271110,GC_Liang:rs2838227|Liang_fwd_tile1-1,.,+,chr21,43270976,A,G,.,PASS,rsid=rs2838227;Region=rs2838227|Liang_fwd_tile...


#### Match this information to the sequences

In [215]:
only_reference_sequences = variant_region_vcf_group[['REF', 'region_chr', 'region_start', 'region_end', 'region_strand']].drop_duplicates(subset=['REF', 'region_chr', 'region_start', 'region_end', 'region_strand'])
pre_metadata_df_group_region_ref = pre_metadata_df_group_variant.merge(only_reference_sequences, left_on=col_name, right_on='REF', how='left')
print(pre_metadata_df_group_region_ref.shape[0])
print(pre_metadata_df_group_region_ref[col_name].nunique())
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref.merge(variant_region_vcf_group[['ALT', 'region_chr', 'region_start', 'region_end', 'region_strand', 'var_pos', 'vcf_REF', 'vcf_ALT']], left_on=col_name, right_on='ALT', how='left')
print(pre_metadata_df_group_region_ref_alt.shape[0])
print(pre_metadata_df_group_region_ref_alt[col_name].nunique())

16
16
16
16


##### Combine x and y columns

In [216]:
pre_metadata_df_group_region_ref_alt = combine_columns(pre_metadata_df_group_region_ref_alt)

In [217]:
pre_metadata_df_group_region_ref_alt
pre_metadata_df_group_region_ref_alt.rename(columns=lambda x: x.replace('region_', '') if 'region_' in x else x , inplace=True)
# Converting float columns to integers
pre_metadata_df_group_region_ref_alt['start'] = pre_metadata_df_group_region_ref_alt['start'].astype(int)
pre_metadata_df_group_region_ref_alt['end'] = pre_metadata_df_group_region_ref_alt['end'].astype(int)

In [218]:
# pre_metadata_df_group_region_ref_alt.head()

#### Compute variant position
- is the variant position 1-based - yes
- chr7:139495297 - start 0-based
https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr7%3A139495450%2D139495452&hgsid=2392258313_huE5stAQaQkB4376zP5rx9IaAEMn

In [219]:
pre_metadata_df_group_region_ref_alt["is_indel"] = pre_metadata_df_group_region_ref_alt.apply(lambda row: find_indel_pattern(row, "vcf_REF", "vcf_ALT"), axis=1)
# add variant_class: SNV or indel
pre_metadata_df_group_region_ref_alt[col_variant_class] = pre_metadata_df_group_region_ref_alt['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')

pre_metadata_df_group_region_ref_alt[col_variant_pos] = pre_metadata_df_group_region_ref_alt.apply(lambda row: get_var_pos(row, var_pos_column='var_pos', allele_column=col_allele, seq_start_column=col_start, variant_1_based=True, start_0_based=True), axis=1)
pre_metadata_df_group_region_ref_alt[[col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]


pre_metadata_df_group_region_ref_alt[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele, 'vcf_REF', 'vcf_ALT', col_variant_class]]
pre_metadata_df_group_region_ref_alt[col_variant_class].value_counts()

pre_metadata_df_group_region_ref_alt['real_ALT'] = pre_metadata_df_group_region_ref_alt.apply(lambda row: get_variant_alternative(row, col_sequence=col_sequence, col_variant_pos=col_variant_pos, col_allele=col_allele), axis=1)
pre_metadata_df_group_region_ref_alt[pre_metadata_df_group_region_ref_alt[col_variant_class] == 'SNV'][[col_chr, 'var_pos', 'vcf_REF', 'vcf_ALT', col_name, col_sequence, col_chr, col_start, col_end, col_strand, 'variant_pos', col_allele,  col_variant_class, 'real_ALT']]


,chr,var_pos,vcf_REF,vcf_ALT,name,sequence,chr,start,end,strand,variant_pos,allele,variant_class,real_ALT
0,chr11,NaN,NaN,NaN,GC_Liang:REF_rs2125358|Liang_fwd_tile1-1,TTTCAATCTTCTCTCCTTGAGAATGAGTCAAAACCCCTTTGACAGC...,chr11,86515771,86516041,+,NA,ref,SNV,NA
1,chr14,NaN,NaN,NaN,GC_Liang:REF_rs17882077|Liang_fwd_tile1-1,CTTCCTCTGAAACCAGCCTGGAGGGAGGAATCAGTTCAGTGCTGAG...,chr14,22848437,22848707,+,NA,ref,SNV,NA
2,chr18,NaN,NaN,NaN,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,CCCAACTTTCAAGGAAGGTGAGAAATGCTGTTTCTTGCTTGAGGCT...,chr18,25019523,25019793,+,NA,ref,SNV,NA
3,chr2,NaN,NaN,NaN,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,CAGAGTTTCTTGTGTAGATTATATTGATTATCTATTACTTACATTA...,chr2,208556887,208557157,+,NA,ref,SNV,NA
4,chr21,NaN,NaN,NaN,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,CTGCAGATAATCAGCTGACGCCCGCCGCGGTGCCTGGGAAACTCAG...,chr21,43270840,43271110,+,NA,ref,SNV,NA
5,chr4,NaN,NaN,NaN,GC_Liang:REF_rs10939614|Liang_fwd_tile1-1,AAAGTCACCTAAACTGGATGCCTGTGGTCACATAACCCAGACACTG...,chr4,9924763,9925033,+,NA,ref,SNV,NA
6,chr7,NaN,NaN,NaN,GC_Liang:REF_rs17603855|Liang_fwd_tile1-1,ACAATTTTTAAGCAGATACACAGACAAAAAGTCCCAAAAGGAAACT...,chr7,138186897,138187167,+,NA,ref,SNV,NA
7,chr7,NaN,NaN,NaN,GC_Liang:REF_rs2530731|Liang_fwd_tile1-1,AAGCAGAAAATCTCTCTTTTTTTGGGGGGGGGGTAATGACTCAATA...,chr7,139495297,139495567,+,NA,ref,SNV,NA
8,chr11,86515916.0,G,C,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...,TTTCAATCTTCTCTCCTTGAGAATGAGTCAAAACCCCTTTGACAGC...,chr11,86515771,86516041,+,144.0,alt,SNV,C
9,chr14,22848616.0,A,G,GC_Liang:ALT_rs17882077|Liang_fwd_tile1-1_rs17...,CTTCCTCTGAAACCAGCCTGGAGGGAGGAATCAGTTCAGTGCTGAG...,chr14,22848437,22848707,+,178.0,alt,SNV,G


### Add SPDI

In [220]:
# add SPDI for alt
pre_metadata_df_group_region_ref_alt['SPDI'] = pre_metadata_df_group_region_ref_alt.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{int(row['var_pos'])}-{row['vcf_REF']}-{row['real_ALT']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_region_df_group.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict

# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict

pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref_alt.apply(lambda row: add_spdi_values_2_reference(row, ref_alt_dict=ref_alt_dict, alt_spdi_dict=alt_spdi_dict, alt_variant_pos_dict=alt_variant_pos_dict, alt_variant_class_dict=alt_variant_class_dict), axis=1)

In [221]:
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref_alt.apply(make_column_arrays, axis = 1)

In [222]:
print('expected number of rows within metadata file:', expected_number)
print('Number of rows in metadata file:', pre_metadata_df_group_region_ref_alt.shape[0])

expected number of rows within metadata file: 16
Number of rows in metadata file: 16


In [223]:
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
pre_metadata_df_group_region_ref_alt[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0

## GC_Mohlke (n: 34)

In [224]:
group_name = 'GC_Mohlke'

In [225]:
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
expected_number = pre_metadata_df_group.shape[0]
variant_region_df_group = variant_region_map.loc[variant_region_map['tmp_label'] == group_name].copy()
region_bed_group = region_bed.loc[region_bed['region_name'].str.contains(group_name)].copy()
vcf_df_group = vcf_df.loc[vcf_df['ID'].str.contains(group_name)].copy()

In [226]:
expected_number # 38

38

In [227]:
vcf_df_group.head()

,CHROM,var_pos,ID,vcf_REF,vcf_ALT,QUAL,FILTER,INFO
4067,chr1,159752293,GC_Mohlke:NC000001_11_159752292_A_G,A,G,.,PASS,Region=NC000001.11|159752292|A|G|MohlkeHepCont...
5229,chr1,230158968,GC_Selvarajan:rs4846913;GC_Mohlke:NC000001_11_...,C,A,.,PASS,rsid=rs4846913;Region=rs4846913|STARR-seq-HepG...
5230,chr1,230159169,GC_Mohlke:NC000001_11_230159168_C_T,C,T,.,PASS,Region=NC000001.11|230158967|C|A|MohlkeHepCont...
5231,chr1,230159329,GC_Mohlke:NC000001_11_230159329_CTTAAAGTGTTCAG...,TCTTAAAGTGTTCAGCACTCCC,T,.,PASS,Region=NC000001.11|230158967|C|A|MohlkeHepCont...
5232,chr1,230161390,GC_Mohlke:NC000001_11_230161389_C_T,C,T,.,PASS,Region=NC000001.11|230161389|C|T|MohlkeHepCont...


In [228]:
variant_region_df_group.head()

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand
46603,GC_Mohlke:NC000001_11_159752292_A_G,GC_Mohlke:NC000001.11|159752292|A|G|MohlkeHepC...,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,GC_Mohlke:ALT_NC000001.11|159752292|A|G|Mohlke...,GC_Mohlke,chr1,159752058,159752328,GC_Mohlke:NC000001.11|159752292|A|G|MohlkeHepC...,.,+
46604,GC_Mohlke:NC000001_11_230158967_C_A,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke,chr1,230158911,230159181,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,.,+
46605,GC_Mohlke:NC000001_11_230159168_C_T,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke,chr1,230159023,230159293,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,.,+
46606,GC_Mohlke:NC000001_11_230159168_C_T,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke,chr1,230159136,230159406,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,.,+
46607,GC_Mohlke:NC000001_11_230159329_CTTAAAGTGTTCAG...,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke,chr1,230159136,230159406,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,.,+


In [229]:
# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])

# if variant related or element
variant_related_list = set(variant_region_df_group['REF'].to_list()).union(set(variant_region_df_group['ALT'].to_list()))
pre_metadata_df_group[col_category] = pre_metadata_df_group[col_name].apply(lambda name: 'variant' if is_variant_related(name, variant_related_list) else 'element')


pre_metadata_df_group[col_class] = pre_metadata_df_group[col_name].apply(lambda name: 'variant negative control' if is_variant_related(name, variant_related_list) else 'element inactive control')
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add allele
alternative_related_list = set(variant_region_df_group['ALT'].to_list())
reference_related_list = set(variant_region_df_group['REF'].to_list())
pre_metadata_df_group[col_allele] = pre_metadata_df_group[col_name].apply(lambda name: 'alt' if is_alternative_related(name, alternative_related_list) else 'ref' if is_reference_related(name, reference_related_list) else 'NA')

pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

In [230]:
# pre_metadata_df_group[col_name].to_list()

### Focus on elements (n: 1) expected 0
- ['GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_headerDuplicate2_2_headerDuplicate1_2']
- again a reference without a variant in the variant map (not sure what this sequence should be)
- What is this sequence? same as the following in the beginning and the finishing
  - `GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3`
- not able to add region information of this sequence


In [231]:
pre_metadata_df_group_element = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'element']
pre_metadata_df_group_element.name.to_list()
# ['GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_headerDuplicate2_2_headerDuplicate1_2']


[]

In [232]:
pre_metadata_df_group_element_region = pre_metadata_df_group_element.merge(region_bed, left_on=col_name, right_on='region_name', how='inner')
pre_metadata_df_group_element_region
# pre_metadata_df_group_element_region.drop(columns=['region_name'], inplace=True)
# pre_metadata_df_group_element_region.columns = [col.split('region_')[1] if 'region_' in col else col for col in pre_metadata_df_group_element_region.columns]

,name,sequence,tmp_label,category,class,source,ref,allele,variant_class,variant_pos,SPDI,info,region_chr,region_start,region_end,region_name,region_score,region_strand


### Focus on variants
- found out that the variant id is named incorrectly (removed duplicates), will add ids with tile information

In [233]:
pre_metadata_df_group_variant = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'variant'].copy()
print(pre_metadata_df_group_variant.shape[0])
pre_metadata_df_group_variant[col_name].nunique()

38


38

In [234]:
print(f'Expected variant number of this group: {variant_region_df_group.shape[0]}')

Expected variant number of this group: 20


In [235]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group, on='ID', how='inner')
print(variant_region_vcf_group.shape[0]) # 16

matchable_names = set(variant_region_vcf_group['ID'].to_list())
all_names = set(vcf_df_group['ID'].to_list())

all_names - matchable_names
# {'GC_Mohlke:NC000001_11_230159168_C_T',
#  'GC_Selvarajan:rs4846913;GC_Mohlke:NC000001_11_230158967_C_A;C_positive_heart_CAD:rs4846913',
#  'GC_Selvarajan:rs603424;GC_Mohlke:NC000010_11_100315721_G_A'}

18


{'GC_Selvarajan:rs4846913;GC_Mohlke:NC000001_11_230158967_C_A;C_positive_heart_CAD:rs4846913',
 'GC_Selvarajan:rs603424;GC_Mohlke:NC000010_11_100315721_G_A'}

#### Investigate headers

In [236]:
pattern = '230159168'
filtered_variant_map = variant_region_df_group.loc[variant_region_df_group['ID'].str.contains(pattern)]
filtered_variant_map['ID'].to_list()

['GC_Mohlke:NC000001_11_230159168_C_T', 'GC_Mohlke:NC000001_11_230159168_C_T']

#### rename ids with tile information

In [237]:
def rename_ids_with_tiles(row, col_name):
    """
    Get a similar ID as the vcf id but because of the bigger tiles the id was repetitive
    """
    name = row[col_name]
    if'GC_Mohlke' not in name:
        return 'NA'

    return 'GC_Mohlke:'+'_'.join(name.split('tile')[-1].split('_')[1:])+('_tile' + name.split('tile')[-1].split('_')[0])

In [238]:
example_string = 'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile2-3_NC000001_11_230159168_C_T'
'_'.join(example_string.split('tile')[-1].split('_')[1:])+('_tile' + example_string.split('tile')[-1].split('_')[0])

# expand: split _tile, expand = true to have a matchable ID with the vcf
variant_region_df_group['new_ID'] = variant_region_df_group.apply(lambda row: rename_ids_with_tiles(row, 'ALT'), axis=1)

# variant_region_df_group[['ID', 'new_ID']]

# variant_region_df_group['ID'] = variant_region_df_group['new_ID'].apply(lambda id: id.split('_tile')[0])

Preprocess the vcf id column to make it matchable

In [239]:
# Function to split the IDs and create new rows while conserving all columns
def split_ids(row, id_col, separator=';'):
    ids = row[id_col].split(separator)
    new_rows = []
    for id in ids:
        new_row = row.copy()
        new_row[id_col] = id
        new_rows.append(new_row)
    return pd.DataFrame(new_rows)

# Apply the function to each row and concatenate the results
vcf_df_group_split = pd.concat(vcf_df_group.apply(lambda row: split_ids(row, id_col='ID'), axis=1).values)

# Reset the index
vcf_df_group_split.reset_index(drop=True, inplace=True)

print(vcf_df_group_split.shape[0]) # 22 before 18

22


In [240]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group_split, on='ID', how='inner')
variant_region_vcf_group.shape[0] # 20

20

In [241]:
variant_region_vcf_group.columns

Index(['ID', 'Region', 'REF', 'ALT', 'tmp_label', 'region_chr', 'region_start',
       'region_end', 'region_name', 'region_score', 'region_strand', 'new_ID',
       'CHROM', 'var_pos', 'vcf_REF', 'vcf_ALT', 'QUAL', 'FILTER', 'INFO'],
      dtype='object')

### Add information to metadata file

In [242]:
only_reference_sequences = variant_region_vcf_group[['REF', 'region_chr', 'region_start', 'region_end', 'region_strand']].drop_duplicates(subset=['REF', 'region_chr', 'region_start', 'region_end', 'region_strand'])
pre_metadata_df_group_region_ref = pre_metadata_df_group_variant.merge(only_reference_sequences, left_on=col_name, right_on='REF', how='left')
print(pre_metadata_df_group_region_ref.shape[0])
print(pre_metadata_df_group_region_ref[col_name].nunique())
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref.merge(variant_region_vcf_group[['ALT', 'region_chr', 'region_start', 'region_end', 'region_strand', 'var_pos', 'vcf_REF', 'vcf_ALT']], left_on=col_name, right_on='ALT', how='left')
print(pre_metadata_df_group_region_ref_alt.shape[0])
print(pre_metadata_df_group_region_ref_alt[col_name].nunique())

38
38
38
38


In [243]:
pre_metadata_df_group_region_ref_alt.columns

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'REF',
       'region_chr_x', 'region_start_x', 'region_end_x', 'region_strand_x',
       'ALT', 'region_chr_y', 'region_start_y', 'region_end_y',
       'region_strand_y', 'var_pos', 'vcf_REF', 'vcf_ALT'],
      dtype='object')

#### Combine columns with _x and _y

In [244]:
# Combine columns
def combine_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith("_x"):
            base_name = col[:-2]  # Remove "_x"
            corresponding_y = base_name + "_y"
            # Combine _x and _y columns
            if corresponding_y in df.columns:
                new_columns[base_name] = df[col].fillna(df[corresponding_y])
            else:
                new_columns[base_name] = df[col]
        elif not col.endswith("_y"):  # Add columns that aren't paired with _x/_y
            new_columns[col] = df[col]
    return pd.DataFrame(new_columns)

# Apply the function
pre_metadata_df_group_region_ref_alt = combine_columns(pre_metadata_df_group_region_ref_alt)

In [245]:
pre_metadata_df_group_region_ref_alt.columns

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'REF',
       'region_chr', 'region_start', 'region_end', 'region_strand', 'ALT',
       'var_pos', 'vcf_REF', 'vcf_ALT'],
      dtype='object')

In [246]:
pre_metadata_df_group_region_ref_alt
pre_metadata_df_group_region_ref_alt.rename(columns=lambda x: x.replace('region_', '') if 'region_' in x else x , inplace=True)
# Converting float columns to integers
pre_metadata_df_group_region_ref_alt['start'] = pre_metadata_df_group_region_ref_alt['start'].astype(int)
pre_metadata_df_group_region_ref_alt['end'] = pre_metadata_df_group_region_ref_alt['end'].astype(int)


In [247]:
# Apply the function to each row and concatenate the results
vcf_df_group_split = pd.concat(vcf_df_group.apply(lambda row: hf.split_ids(row, id_col='ID'), axis=1).values)

# Reset the index
vcf_df_group_split.reset_index(drop=True, inplace=True)

print(vcf_df_group_split.shape[0]) # 22 before 19

22


In [248]:
vcf_df_group.shape[0]

19

In [249]:
pre_metadata_df_group_region_ref_alt.head()

,name,sequence,tmp_label,category,class,source,ref,allele,variant_class,variant_pos,...,info,REF,chr,start,end,strand,ALT,var_pos,vcf_REF,vcf_ALT
0,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,AGTGTGTCTGAGCAGTGCCCCAGCCCCCATGCCGCTTTGGATTTCA...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,chr1,230158911,230159181,+,NaN,NaN,NaN,NaN
1,GC_Mohlke:REF_NC000010.11|100315721|G|A|Mohlke...,TAATAAATATATTACCAGTCTGATTTTCTTTGTAGGGGAAAGGGGG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mohlke:REF_NC000010.11|100315721|G|A|Mohlke...,chr10,100315633,100315903,+,NaN,NaN,NaN,NaN
2,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,AGTGTGTCTGAGCAGTGCCCCAGCCCCCATGCCGCTTTGGATTTCA...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,,NaN,chr1,230158911,230159181,+,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,230158968.0,C,A
3,GC_Mohlke:ALT_NC000010.11|100315721|G|A|Mohlke...,TAATAAATATATTACCAGTCTGATTTTCTTTGTAGGGGAAAGGGGG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,,NaN,chr10,100315633,100315903,+,GC_Mohlke:ALT_NC000010.11|100315721|G|A|Mohlke...,100315722.0,G,A
4,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,chr1,159752058,159752328,+,NaN,NaN,NaN,NaN


#### Compute variant position
- is the variant position 1-based - yes
- start is 0 based: https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A230158911%2D230158931&hgsid=2391760529_1w3aWXOGHd4oNaG5x6ZVm2Jd2mVF
- what about indels: 1-based 193

In [ ]:
import re
def find_indel_pattern(row, ref_column, alt_column):
    """Check if in ref or alt is more than 1 subsequent nucleotide indicating an indel
    Special case: I want it to check for the notation of muliallelic variants as well (A,T) if any of these is an indel"""
    if hf.is_alternative(row[col_allele]):
        pattern = r'(^[ACGT]{2,})|(,[ACGT]{2,})'
        # Check if the text matches the pattern
        ref_indel = bool(re.match(pattern, row[ref_column]))
        alt_indel = bool(re.match(pattern, row[alt_column]))
        return ref_indel or alt_indel
    else: False


In [251]:
pre_metadata_df_group_region_ref_alt["is_indel"] = pre_metadata_df_group_region_ref_alt.apply(lambda row: find_indel_pattern(row, "vcf_REF", "vcf_ALT"), axis=1)
# add variant_class: SNV or indel
pre_metadata_df_group_region_ref_alt[col_variant_class] = pre_metadata_df_group_region_ref_alt['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')

pre_metadata_df_group_region_ref_alt[col_variant_pos] = pre_metadata_df_group_region_ref_alt.apply(lambda row: get_var_pos(row, var_pos_column='var_pos', allele_column=col_allele, seq_start_column=col_start, variant_1_based=True, start_0_based=True), axis=1)
pre_metadata_df_group_region_ref_alt[[col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]


pre_metadata_df_group_region_ref_alt[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele, 'vcf_REF', 'vcf_ALT', col_variant_class]]
pre_metadata_df_group_region_ref_alt[col_variant_class].value_counts()


variant_class
SNV      37
indel     1
Name: count, dtype: int64

In [252]:
pre_metadata_df_group_region_ref_alt['real_ALT'] = pre_metadata_df_group_region_ref_alt.apply(lambda row: get_variant_alternative(row, col_sequence=col_sequence, col_variant_pos=col_variant_pos, col_allele=col_allele), axis=1)
pre_metadata_df_group_region_ref_alt[pre_metadata_df_group_region_ref_alt[col_variant_class] == 'SNV'][[col_chr, 'var_pos', 'vcf_REF', 'vcf_ALT', col_name, col_sequence, col_chr, col_start, col_end, col_strand, 'variant_pos', col_allele,  col_variant_class, 'real_ALT']]


,chr,var_pos,vcf_REF,vcf_ALT,name,sequence,chr,start,end,strand,variant_pos,allele,variant_class,real_ALT
0,chr1,NaN,NaN,NaN,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,AGTGTGTCTGAGCAGTGCCCCAGCCCCCATGCCGCTTTGGATTTCA...,chr1,230158911,230159181,+,NA,ref,SNV,NA
1,chr10,NaN,NaN,NaN,GC_Mohlke:REF_NC000010.11|100315721|G|A|Mohlke...,TAATAAATATATTACCAGTCTGATTTTCTTTGTAGGGGAAAGGGGG...,chr10,100315633,100315903,+,NA,ref,SNV,NA
2,chr1,230158968.0,C,A,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,AGTGTGTCTGAGCAGTGCCCCAGCCCCCATGCCGCTTTGGATTTCA...,chr1,230158911,230159181,+,56.0,alt,SNV,A
3,chr10,100315722.0,G,A,GC_Mohlke:ALT_NC000010.11|100315721|G|A|Mohlke...,TAATAAATATATTACCAGTCTGATTTTCTTTGTAGGGGAAAGGGGG...,chr10,100315633,100315903,+,88.0,alt,SNV,A
4,chr1,NaN,NaN,NaN,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,chr1,159752058,159752328,+,NA,ref,SNV,NA
5,chr1,NaN,NaN,NaN,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,chr1,230159023,230159293,+,NA,ref,SNV,NA
6,chr1,NaN,NaN,NaN,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,chr1,230159136,230159406,+,NA,ref,SNV,NA
8,chr1,NaN,NaN,NaN,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,chr1,230161269,230161539,+,NA,ref,SNV,NA
9,chr10,NaN,NaN,NaN,GC_Mohlke:REF_NC000010.11|12265894|C|T|MohlkeH...,TGTGGAAAGGTAGCTGTGGGAAGGGACGTAGCCTTTGTAATGCAAG...,chr10,12265719,12265989,+,NA,ref,SNV,NA
10,chr13,NaN,NaN,NaN,GC_Mohlke:REF_NC000013.11|94602127|A|G|MohlkeH...,GAGCCGAGGCGTCGGTGCAGACCTGGAGACGGGCATGGGGGGGCTG...,chr13,94601893,94602163,+,NA,ref,SNV,NA


#### Add SPDI

In [253]:
# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

In [254]:
# add SPDI for alt
pre_metadata_df_group_region_ref_alt['SPDI'] = pre_metadata_df_group_region_ref_alt.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{int(row['var_pos'])}-{row['vcf_REF']}-{row['real_ALT']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_region_df_group.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict

# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict

pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref_alt.apply(lambda row: add_spdi_values_2_reference(row, ref_alt_dict=ref_alt_dict, alt_spdi_dict=alt_spdi_dict, alt_variant_pos_dict=alt_variant_pos_dict, alt_variant_class_dict=alt_variant_class_dict), axis=1)

In [255]:
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref_alt.apply(make_column_arrays, axis = 1)

In [256]:
print('expected number of rows within metadata file:', expected_number)
print('Number of rows in metadata file:', pre_metadata_df_group_region_ref_alt.shape[0])

expected number of rows within metadata file: 38
Number of rows in metadata file: 38


In [257]:
pre_metadata_df_group_region_ref_alt

,name,sequence,tmp_label,category,class,source,ref,allele,variant_class,variant_pos,...,chr,start,end,strand,ALT,var_pos,vcf_REF,vcf_ALT,is_indel,real_ALT
0,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,AGTGTGTCTGAGCAGTGCCCCAGCCCCCATGCCGCTTTGGATTTCA...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[ref],[SNV],[56],...,chr1,230158911,230159181,+,NaN,NaN,NaN,NaN,None,NA
1,GC_Mohlke:REF_NC000010.11|100315721|G|A|Mohlke...,TAATAAATATATTACCAGTCTGATTTTCTTTGTAGGGGAAAGGGGG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[ref],[SNV],[88],...,chr10,100315633,100315903,+,NaN,NaN,NaN,NaN,None,NA
2,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,AGTGTGTCTGAGCAGTGCCCCAGCCCCCATGCCGCTTTGGATTTCA...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[alt],[SNV],[56],...,chr1,230158911,230159181,+,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,230158968.0,C,A,False,A
3,GC_Mohlke:ALT_NC000010.11|100315721|G|A|Mohlke...,TAATAAATATATTACCAGTCTGATTTTCTTTGTAGGGGAAAGGGGG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[alt],[SNV],[88],...,chr10,100315633,100315903,+,GC_Mohlke:ALT_NC000010.11|100315721|G|A|Mohlke...,100315722.0,G,A,False,A
4,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[ref],[SNV],[234],...,chr1,159752058,159752328,+,NaN,NaN,NaN,NaN,None,NA
5,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[ref],[SNV],[145],...,chr1,230159023,230159293,+,NaN,NaN,NaN,NaN,None,NA
6,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,"[ref, ref]","[SNV, indel]","[32, 192]",...,chr1,230159136,230159406,+,NaN,NaN,NaN,NaN,None,NA
7,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[alt],[indel],[192],...,chr1,230159136,230159406,+,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,230159329.0,TCTTAAAGTGTTCAGCACTCCC,T,True,T
8,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[ref],[SNV],[120],...,chr1,230161269,230161539,+,NaN,NaN,NaN,NaN,None,NA
9,GC_Mohlke:REF_NC000010.11|12265894|C|T|MohlkeH...,TGTGGAAAGGTAGCTGTGGGAAGGGACGTAGCCTTTGTAATGCAAG...,GC_Mohlke,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,[ref],[SNV],[175],...,chr10,12265719,12265989,+,NaN,NaN,NaN,NaN,None,NA


In [258]:
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
pre_metadata_df_group_region_ref_alt[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0

## GC_Selvarajan (n: 364)

In [15]:
group_name = 'GC_Selvarajan'

In [16]:
def is_variant_related(name, variant_related_list):
    return name in variant_related_list

def is_reference_related(name, reference_related_list):
    return name in reference_related_list

def is_alternative_related(name, variant_related_list):
    return name in variant_related_list


In [17]:
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
expected_number = pre_metadata_df_group.shape[0]
variant_region_df_group = variant_region_map.loc[variant_region_map['tmp_label'] == group_name].copy()
vcf_df_group = vcf_df.loc[vcf_df['ID'].str.contains(group_name)].copy()

In [18]:
# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])

# if variant related or element
variant_related_list = set(variant_region_df_group['REF'].to_list()).union(set(variant_region_df_group['ALT'].to_list()))
pre_metadata_df_group[col_category] = pre_metadata_df_group[col_name].apply(lambda name: 'variant' if is_variant_related(name, variant_related_list) else 'element')


pre_metadata_df_group[col_class] = pre_metadata_df_group[col_name].apply(lambda name: 'variant negative control' if is_variant_related(name, variant_related_list) else 'element inactive control')
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add allele
alternative_related_list = set(variant_region_df_group['ALT'].to_list())
reference_related_list = set(variant_region_df_group['REF'].to_list())
pre_metadata_df_group[col_allele] = pre_metadata_df_group[col_name].apply(lambda name: 'alt' if is_alternative_related(name, alternative_related_list) else 'ref' if is_reference_related(name, reference_related_list) else 'NA')

pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

In [19]:
pre_metadata_df_group[col_category].value_counts()

category
variant    361
element      3
Name: count, dtype: int64

In [20]:
print('Number of variants', variant_region_df_group.shape[0])
print('Number of ref: ', variant_region_df_group['REF'].nunique()) # 163
print('Number of alt: ', variant_region_df_group['ALT'].nunique()) # 198

Number of variants 198
Number of ref:  163
Number of alt:  198


In [21]:
variant_related_list - set(pre_metadata_df_group[col_name].to_list())

set()

#### Focus on elements
- find why elements have REF_ and ALT_ tag but are not in the variant map
- GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787 cannot be found in region.bed
- ['GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787']

In [266]:
# TODO find why elements have REF_ and ALT_ tag but are not in the variant map

In [22]:
pre_metadata_df_group_element = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'element'].copy()
pre_metadata_df_group_element.name.to_list()


['GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787']

#### process to match them:
- split by '#'
- change the header and remove the 'ALT_' or 'REF_' from the headers and match again with the region bed


In [268]:
# # Apply the function to each row and concatenate the results
# pre_metadata_df_group_element_split = pd.concat(pre_metadata_df_group_element.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# # Reset the index
# pre_metadata_df_group_element_split.reset_index(drop=True, inplace=True)
# pre_metadata_df_group_element = pre_metadata_df_group_element_split.copy()
# print(pre_metadata_df_group_element.shape[0])

In [23]:
def remove_alt_ref_pattern_selvarajan(name):
    if ':REF_' in name:
        return name.replace(':REF_', ':')
    elif ':ALT_' in name:
        name = name.replace(':ALT_', ':')
        return name.split('_rs')[0]

In [24]:
pre_metadata_df_group_element['new_region_map_names'] = pre_metadata_df_group_element[col_name].apply(remove_alt_ref_pattern_selvarajan)

In [25]:
pre_metadata_df_group_element.new_region_map_names.to_list()

['GC_Selvarajan:rs216222|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:rs754064|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:rs2297787|STARR-seq-HepG2_fwd_tile1-1']

In [26]:
pre_metadata_df_group_element_region = pre_metadata_df_group_element.merge(region_bed, left_on='new_region_map_names', right_on='region_name', how='inner')
pre_metadata_df_group_element_region.drop(columns=['region_name'], inplace=True)
pre_metadata_df_group_element_region.columns = [col.split('region_')[1] if 'region_' in col else col for col in pre_metadata_df_group_element_region.columns]

In [27]:
print('Number of sequences which should be matchable with the region.bed:', pre_metadata_df_group_element.loc[pre_metadata_df_group_element[col_name].str.startswith(group_name)].shape[0])


Number of sequences which should be matchable with the region.bed: 3


In [28]:
print('Number of sequences which are matchable with the region.bed:', pre_metadata_df_group_element_region.shape[0])

Number of sequences which are matchable with the region.bed: 3


In [29]:
pre_metadata_df_group_element_region_final = pre_metadata_df_group_element_region[interesting_columns].copy()
pre_metadata_df_group_element_region_final.shape[0]

3

In [31]:
pre_metadata_df_group_element_region_final

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd...,TGCCTATAGTCCCAGCTACTCAGGAGGCTGAGGCAGGAGAATTGCT...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr17,2252271,2252541,+,NA,NA,NA,NA,
1,GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd...,CTGGTGTTATATTCACCTGACTAAATTTAGAAGGAACTGTGTATCT...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr20,48798773,48799043,+,NA,NA,NA,NA,
2,GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fw...,GTGGCGTGGATTTGGGGATGGATTGAACTAGTAAGTGCTACTAGGT...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr10,102920244,102920514,+,NA,NA,NA,NA,


#### Focus on variants

In [30]:
pre_metadata_df_group_variant = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'variant'].copy()
print(pre_metadata_df_group_variant.shape[0])
pre_metadata_df_group_variant[col_name].nunique()

361


361

In [32]:
pre_metadata_df_group_region_alt = pre_metadata_df_group_variant.merge(variant_region_map[['ID', 'ALT', 'region_chr', 'region_start', 'region_end', 'region_strand']], left_on=col_name, right_on='ALT', how='left')
print(pre_metadata_df_group_region_alt.shape[0])
print(pre_metadata_df_group_region_alt[col_name].nunique())
only_reference_sequences = variant_region_map[['ID', 'REF', 'region_chr', 'region_start', 'region_end', 'region_strand']].drop_duplicates()
pre_metadata_df_group_region_alt_ref = pre_metadata_df_group_region_alt.merge(only_reference_sequences, left_on=col_name, right_on='REF', how='left')
print(pre_metadata_df_group_region_alt_ref.shape[0])
print(pre_metadata_df_group_region_alt_ref[col_name].nunique())

361
361
396
361


In [33]:
pre_metadata_df_group_region_alt_ref.columns

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'ID_x', 'ALT',
       'region_chr_x', 'region_start_x', 'region_end_x', 'region_strand_x',
       'ID_y', 'REF', 'region_chr_y', 'region_start_y', 'region_end_y',
       'region_strand_y'],
      dtype='object')

### Combine columns with _x and _y

In [279]:
# Combine columns
def combine_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith("_x"):
            base_name = col[:-2]  # Remove "_x"
            corresponding_y = base_name + "_y"
            # Combine _x and _y columns
            if corresponding_y in df.columns:
                new_columns[base_name] = df[col].fillna(df[corresponding_y])
            else:
                new_columns[base_name] = df[col]
        elif not col.endswith("_y"):  # Add columns that aren't paired with _x/_y
            new_columns[col] = df[col]
    return pd.DataFrame(new_columns)

# Apply the function
pre_metadata_df_group_region_alt_ref = combine_columns(pre_metadata_df_group_region_alt_ref)

In [280]:
pre_metadata_df_group_region_alt_ref
pre_metadata_df_group_region_alt_ref.rename(columns=lambda x: x.replace('region_', '') if 'region_' in x else x , inplace=True)
# Converting float columns to integers
pre_metadata_df_group_region_alt_ref['start'] = pre_metadata_df_group_region_alt_ref['start'].astype(int)
pre_metadata_df_group_region_alt_ref['end'] = pre_metadata_df_group_region_alt_ref['end'].astype(int)


In [281]:
# remove duplicates:
pre_metadata_df_group_region_alt_ref_dedup = pre_metadata_df_group_region_alt_ref.drop_duplicates(subset=interesting_columns)
pre_metadata_df_group_region_alt_ref_dedup.shape[0]

361

##### Old way of doing it: generate the new headers: new way use different variant region map

In [282]:
# # need to extract the rsid because the variant vcf has a different header than the variant id
# # Function to extract rsid
# def extract_rsid(s):
#     import re
#     match = re.findall(r"rs\d+", s)
#     return match[-1] if match else None


# pre_metadata_df_group_region_alt_ref['rsid'] = pre_metadata_df_group_region_alt_ref[col_name].apply(extract_rsid)

In [283]:
# vcf_df_group['rsid'] = vcf_df_group['ID'].apply(extract_rsid)
# vcf_df_group.columns
# # merge
# print(pre_metadata_df_group_region_alt_ref.shape[0])
# print('unique values before merge: ', pre_metadata_df_group_region_alt_ref['name'].nunique())
# variant_pre_metadata_vcf = pre_metadata_df_group_region_alt_ref.merge(vcf_df_group[['rsid', 'var_pos', 'vcf_REF', 'vcf_ALT']], on='rsid', how='inner')
# print(variant_pre_metadata_vcf.shape[0]) # 361
# print('unique values: ', pre_metadata_df_group_region_alt_ref['name'].nunique())

In [284]:
# vcf_df_group.head()

In [285]:
pre_metadata_df_group_region_alt_ref_dedup.head()

,name,sequence,tmp_label,category,class,source,ref,allele,variant_class,variant_pos,SPDI,info,ID,ALT,chr,start,end,strand,REF
0,"GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2,rs...",GGGGATGGTGCGGCCACGTCCAGGAGGCAGAAGCACAGAGAATGTT...,GC_Selvarajan,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,NA,,GC_Selvarajan:rs9660819,NaN,chr1,3036131,3036401,+,"GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2,rs..."
2,"GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2,rs...",GGGGTTGTGTGGTGTGTGTATCATGGTGTGTGTATATGTACTCTGT...,GC_Selvarajan,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,NA,,GC_Selvarajan:rs72856440,NaN,chr1,3039882,3040152,+,"GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2,rs..."
4,"GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2,rs...",CTGTGCATACGGGCTGTGTATATATATGCTGTGTGTATATGTTGCA...,GC_Selvarajan,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,NA,,GC_Selvarajan:rs72856440,NaN,chr1,3039924,3040194,+,"GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2,rs..."
6,"GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2,rs...",TGCATATGTGTTGTGTGTATGGGCATTGCATGTTGTATGTATGGGG...,GC_Selvarajan,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,NA,,GC_Selvarajan:rs72856440,NaN,chr1,3039966,3040236,+,"GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2,rs..."
8,GC_Selvarajan:REF_rs2493288|STARR-seq-HepG2_fw...,GGGTGCTGGCAGCCAGGGGTCTTTGAAGGCCCTCGAGGGGACCTTG...,GC_Selvarajan,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,NA,,GC_Selvarajan:rs2493288,NaN,chr1,3414235,3414505,+,GC_Selvarajan:REF_rs2493288|STARR-seq-HepG2_fw...


##### New way use the ID column

In [286]:
# split by ids:
# Apply the function to each row and concatenate the results
vcf_df_group_split = pd.concat(vcf_df_group.apply(lambda row: split_ids(row, id_col='ID'), axis=1).values)

# Reset the index
vcf_df_group_split.reset_index(drop=True, inplace=True)
print(vcf_df_group.shape[0])
print(vcf_df_group_split.shape[0]) # 181 before 170

170
181


In [287]:
variant_pre_metadata_vcf = pre_metadata_df_group_region_alt_ref_dedup.merge(vcf_df_group_split[['ID', 'var_pos', 'vcf_REF', 'vcf_ALT']], on='ID', how='left')
variant_pre_metadata_vcf.shape[0]
# 361

361

In [288]:
pre_metadata_df_group_region_alt_ref_dedup.shape[0]

361

In [289]:
pre_metadata_df_group_region_alt_ref_dedup_names = set(pre_metadata_df_group_region_alt_ref_dedup['ID'].to_list())
vcf_df_group_names = set(vcf_df_group_split['ID'].to_list())
pre_metadata_df_group_region_alt_ref_dedup_names - vcf_df_group_names

set()

In [290]:
vcf_df_group_names - pre_metadata_df_group_region_alt_ref_dedup_names

{'C_positive_heart_CAD:rs12721051',
 'C_positive_heart_CAD:rs12740374',
 'C_positive_heart_CAD:rs17293632',
 'C_positive_heart_CAD:rs4846913',
 'GC_Kircher:NC000001_11_109274967_G_A',
 'GC_Mohlke:NC000001_11_230158967_C_A',
 'GC_Mohlke:NC000010_11_100315721_G_A',
 'cardiac_neuro_cava_random:BRCA2|ENSG00000139618.18|EH38E1665846|13-32424473-T-G',
 'cardiac_neuro_cava_random:CDKN2B|ENSG00000147883.12|EH38E2687344|9-22052735-T-C',
 'cardiac_neuro_cava_random:PLPP3|ENSG00000162407.9|EH38E2813743|1-56482618-A-G',
 'cardiac_neuro_cava_random:SMAD3|ENSG00000166949.17|EH38E3141606|15-67150258-C-T'}

In [291]:
variant_pre_metadata_vcf['var_pos'].isna().sum()

0

#### Compute variant position
- is the variant position 1-based? (look into rsid => variant position and SPDI (0-based))
- https://www.ncbi.nlm.nih.gov/snp/?term=rs9661525 (1-based)

In [ ]:
import re
def find_indel_pattern(row, ref_column, alt_column):
    """Check if in ref or alt is more than 1 subsequent nucleotide indicating an indel
    Special case: I want it to check for the notation of muliallelic variants as well (A,T) if any of these is an indel"""
    pattern = r'(^[ACGT]{2,})|(,[ACGT]{2,})'
    # Check if the text matches the pattern
    ref_indel = bool(re.match(pattern, row[ref_column]))
    alt_indel = bool(re.match(pattern, row[alt_column]))
    return ref_indel or alt_indel


def get_variant_alternative(row, col_sequence, col_variant_pos, col_allele, col_variant_class='variant_class'):
    """Return the char at the variant pos position"""
    if not hf.is_alternative(row[col_allele]):
        return pd.NA
    if row[col_variant_class] == 'SNV':
        variant_position = int(row[col_variant_pos])
        return row[col_sequence][variant_position]
    elif row[col_variant_class] == 'indel':
        if ',' in row['vcf_ALT']:
            raise ValueError('Special case of indel. Please check manually')
        return row['vcf_ALT']
    return pd.NA

In [293]:
variant_pre_metadata_vcf[col_variant_pos] = variant_pre_metadata_vcf.apply(lambda row: get_var_pos(row, var_pos_column='var_pos', allele_column=col_allele, seq_start_column=col_start, variant_1_based=True, start_0_based=True), axis=1)
variant_pre_metadata_vcf[[col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]

# add variant_class: SNV or indel
variant_pre_metadata_vcf["is_indel"] = variant_pre_metadata_vcf.apply(lambda row: find_indel_pattern(row, "vcf_REF", "vcf_ALT"), axis=1)
variant_pre_metadata_vcf[col_variant_class] = variant_pre_metadata_vcf['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')
variant_pre_metadata_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele, 'vcf_REF', 'vcf_ALT', col_variant_class]]
variant_pre_metadata_vcf[col_variant_class].value_counts() # 2 indels
# variant_pre_metadata_vcf[variant_pre_metadata_vcf[col_variant_class] == 'indel'][[col_chr, 'var_pos', col_name, 'vcf_REF', 'vcf_ALT', 'rsid', col_sequence, col_chr, col_start, col_end, 'variant_pos', col_allele,  col_variant_class]]
# variant_pre_metadata_vcf[variant_pre_metadata_vcf[col_variant_class] == 'indel'][col_sequence].to_list()[0][172] # G
# variant_pre_metadata_vcf[variant_pre_metadata_vcf[col_variant_class] == 'indel'][col_sequence].to_list()[1][172] # T

variant_class
SNV    361
Name: count, dtype: int64

##### Learned from an example: vcf_ALT is not correctly set, use variant_pos and sequence


In [294]:
variant_pre_metadata_vcf['real_ALT'] = variant_pre_metadata_vcf.apply(lambda row: get_variant_alternative(row, col_sequence=col_sequence, col_variant_pos=col_variant_pos, col_allele=col_allele), axis=1)

- Example if the variant position is correct: rs6475604, rs72856440
- checked by hand 

In [295]:
# variant_pre_metadata_vcf[variant_pre_metadata_vcf['var_pos'] == 14292721] # 2 alt
# variant_pre_metadata_vcf[variant_pre_metadata_vcf['rsid'] == 'rs6475604'][[col_name,'rsid', col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]] # 104 ref and alt
# # variant_pre_metadata_vcf[variant_pre_metadata_vcf['rsid'] == 'rs6475604'][col_sequence].to_list()
# variant_pre_metadata_vcf[variant_pre_metadata_vcf['rsid'] == 'rs72856440'][[col_name, 'rsid', col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]] # 3x ref and alt
# # var pos: 135, 93, 51
# variant_pre_metadata_vcf[variant_pre_metadata_vcf['rsid'] == 'rs72856440'][col_sequence].to_list()

In [296]:
# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

In [297]:
# add SPDI for alt
variant_pre_metadata_vcf['SPDI'] = variant_pre_metadata_vcf.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{row['var_pos']}-{row['vcf_REF']}-{row['real_ALT']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_region_df_group.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = variant_pre_metadata_vcf.loc[variant_pre_metadata_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = variant_pre_metadata_vcf.loc[variant_pre_metadata_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict

# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = variant_pre_metadata_vcf.loc[variant_pre_metadata_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict

variant_pre_metadata_vcf = variant_pre_metadata_vcf.apply(lambda row: add_spdi_values_2_reference(row, ref_alt_dict=ref_alt_dict, alt_spdi_dict=alt_spdi_dict, alt_variant_pos_dict=alt_variant_pos_dict, alt_variant_class_dict=alt_variant_class_dict), axis=1)
variant_pre_metadata_vcf_final = variant_pre_metadata_vcf[interesting_columns].copy()

In [298]:
variant_pre_metadata_vcf_final = variant_pre_metadata_vcf_final.apply(make_column_arrays, axis = 1)


In [299]:
variant_pre_metadata_vcf_final = variant_pre_metadata_vcf_final.reset_index()
pre_metadata_df_group_element_region = pre_metadata_df_group_element_region.reset_index()

In [300]:
# variant_pre_metadata_vcf_final.shape[0] # 361
variant_pre_metadata_vcf_final[col_name].nunique()

361

In [301]:
pre_metadata_df_group_element_region_final.head()

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd...,TGCCTATAGTCCCAGCTACTCAGGAGGCTGAGGCAGGAGAATTGCT...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr17,2252271,2252541,+,NA,NA,NA,NA,
1,GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd...,CTGGTGTTATATTCACCTGACTAAATTTAGAAGGAACTGTGTATCT...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr20,48798773,48799043,+,NA,NA,NA,NA,
2,GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fw...,GTGGCGTGGATTTGGGGATGGATTGAACTAGTAAGTGCTACTAGGT...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr10,102920244,102920514,+,NA,NA,NA,NA,


In [302]:
combined_metadata_final = pd.concat([pre_metadata_df_group_element_region_final, variant_pre_metadata_vcf_final], ignore_index=True)

In [303]:
### Merge variant and element results
print('expected number: ', expected_number)
print('number of rows in the metadata file: ', combined_metadata_final.shape[0])

expected number:  364
number of rows in the metadata file:  364


In [304]:
duplicated = combined_metadata_final.loc[combined_metadata_final.duplicated(subset=[col_name], keep=False)]
duplicated

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,index


In [305]:
# combined_metadata_final.head()

In [306]:
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
combined_metadata_final[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0

### Processes 
- variant informtion
- genomic location
- SPDI

## GC_Atrial_fib (n: 45)

In [307]:
group_name = 'GC_Atrial_fib'

pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
expected_number = pre_metadata_df_group.shape[0]
variant_region_df_group = variant_region_map.loc[variant_region_map['tmp_label'] == group_name].copy()
vcf_df_group = vcf_df.loc[vcf_df['ID'].str.contains(group_name)].copy()
region_bed_group = region_bed.loc[region_bed['region_name'].str.startswith(group_name)].copy()
print('expected_number: ', expected_number)

expected_number:  45


In [308]:
pre_metadata_df_group.head()

,name,sequence,tmp_label
73940,"GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF,rs78...",AGGACCGGATCAACTTCATTTCATTATAATCAAAAAGGATTTTTAA...,GC_Atrial_fib
73941,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,GC_Atrial_fib
73942,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,GC_Atrial_fib
73943,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,GC_Atrial_fib
73944,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,GC_Atrial_fib


In [309]:
variant_region_df_group.head()

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand
46374,GC_Atrial_fib:rs74541936,GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib,chr1,154813248,154813518,GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fw...,.,+
46375,GC_Atrial_fib:rs34292822,GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib,chr1,154839744,154840014,GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fw...,.,+
46376,GC_Atrial_fib:rs12754189,GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib,chr1,154840018,154840288,GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fw...,.,+
46377,GC_Atrial_fib:rs36088503,GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib,chr1,154840331,154840601,GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fw...,.,+
46378,GC_Atrial_fib:rs76749863,"GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs7...",GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib,chr1,154860467,154860737,"GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs7...",.,+


In [310]:
# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])

# if variant related or element
variant_related_list = set(variant_region_df_group['REF'].to_list()).union(set(variant_region_df_group['ALT'].to_list()))
pre_metadata_df_group[col_category] = pre_metadata_df_group[col_name].apply(lambda name: 'variant' if is_variant_related(name, variant_related_list) else 'element')


pre_metadata_df_group[col_class] = pre_metadata_df_group[col_name].apply(lambda name: 'variant negative control' if is_variant_related(name, variant_related_list) else 'element inactive control')
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add allele
alternative_related_list = set(variant_region_df_group['ALT'].to_list())
reference_related_list = set(variant_region_df_group['REF'].to_list())
pre_metadata_df_group[col_allele] = pre_metadata_df_group[col_name].apply(lambda name: 'alt' if is_alternative_related(name, alternative_related_list) else 'ref' if is_reference_related(name, reference_related_list) else 'NA')

pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

In [311]:
# check number of variants and elements
pre_metadata_df_group[col_category].value_counts()

category
variant    44
element     1
Name: count, dtype: int64

### Focus on Elements

In [312]:
pre_metadata_df_group_element = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'element'].copy()
pre_metadata_df_group_element.name.to_list()


['GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF,rs7811851|CAV1|STARR-seq-AF_fwd_tile2-3']

In [313]:
pre_metadata_df_group_element_region = pre_metadata_df_group_element.merge(region_bed_group, left_on=col_name, right_on='region_name', how='left').copy()
pre_metadata_df_group_element_region.drop(columns=['region_name'], inplace=True)
pre_metadata_df_group_element_region.columns = [col.split('region_')[1] if 'region_' in col else col for col in pre_metadata_df_group_element_region.columns]

In [314]:
pre_metadata_df_group_element_region_final = pre_metadata_df_group_element_region[interesting_columns]

In [315]:
pre_metadata_df_group_element_region_final.shape[0]

1

### Focus on Variants
- found out that the variant id is named incorrectly (removed duplicates), will add ids with tile information

In [316]:
pre_metadata_df_group_variant = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'variant'].copy()
print(pre_metadata_df_group_variant.shape[0])
pre_metadata_df_group_variant[col_name].nunique()

44


44

In [317]:
print(f'Expected variant number of this group: {variant_region_df_group.shape[0]}')

Expected variant number of this group: 23


In [318]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group, on='ID', how='inner')
print(variant_region_vcf_group.shape[0]) # 21

matchable_names = set(variant_region_vcf_group['ID'].to_list())
all_names = set(vcf_df_group['ID'].to_list())

all_names - matchable_names
# {'cardiac_neuro_cava_random:HCN4|ENSG00000138622.4|EH38E1776818|15-73390350-T-C;GC_Atrial_fib:rs6495062',
#  'cardiac_neuro_cava_random:HCN4|ENSG00000138622.4|EH38E1776818|15-73390367-G-T;GC_Atrial_fib:rs6495063'}

21


{'cardiac_neuro_cava_random:HCN4|ENSG00000138622.4|EH38E1776818|15-73390350-T-C;GC_Atrial_fib:rs6495062',
 'cardiac_neuro_cava_random:HCN4|ENSG00000138622.4|EH38E1776818|15-73390367-G-T;GC_Atrial_fib:rs6495063'}

split the vcfs ids by ";"

In [319]:
# Function to split the IDs and create new rows while conserving all columns
def split_ids(row, id_col):
    ids = row[id_col].split(';')
    new_rows = []
    for id in ids:
        new_row = row.copy()
        new_row[id_col] = id
        new_rows.append(new_row)
    return pd.DataFrame(new_rows)

# Apply the function to each row and concatenate the results
vcf_df_group_split = pd.concat(vcf_df_group.apply(lambda row: split_ids(row, id_col='ID'), axis=1).values)

# Reset the index
vcf_df_group_split.reset_index(drop=True, inplace=True)

print(vcf_df_group_split.shape[0]) # 22 before 18

25


In [320]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group_split, on='ID', how='inner')
variant_region_vcf_group.shape[0] # 23

23

### Add information to metadata file: region and variant information

In [321]:
only_reference_sequences = variant_region_vcf_group[['REF', 'region_chr', 'region_start', 'region_end', 'region_strand']].drop_duplicates(subset=['REF', 'region_chr', 'region_start', 'region_end', 'region_strand'])
pre_metadata_df_group_region_ref = pre_metadata_df_group_variant.merge(only_reference_sequences, left_on=col_name, right_on='REF', how='left')
print(pre_metadata_df_group_region_ref.shape[0])
print(pre_metadata_df_group_region_ref[col_name].nunique())
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref.merge(variant_region_vcf_group[['ALT', 'region_chr', 'region_start', 'region_end', 'region_strand', 'var_pos', 'vcf_REF', 'vcf_ALT']], left_on=col_name, right_on='ALT', how='left')
print(pre_metadata_df_group_region_ref_alt.shape[0])
print(pre_metadata_df_group_region_ref_alt[col_name].nunique())

44
44
44
44


In [322]:
pre_metadata_df_group_region_ref_alt.columns

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'REF',
       'region_chr_x', 'region_start_x', 'region_end_x', 'region_strand_x',
       'ALT', 'region_chr_y', 'region_start_y', 'region_end_y',
       'region_strand_y', 'var_pos', 'vcf_REF', 'vcf_ALT'],
      dtype='object')

#### Combine _x and _y columns (regions from ref and alt matching)

In [323]:
# Combine columns
def combine_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith("_x"):
            base_name = col[:-2]  # Remove "_x"
            corresponding_y = base_name + "_y"
            # Combine _x and _y columns
            if corresponding_y in df.columns:
                new_columns[base_name] = df[col].fillna(df[corresponding_y])
            else:
                new_columns[base_name] = df[col]
        elif not col.endswith("_y"):  # Add columns that aren't paired with _x/_y
            new_columns[col] = df[col]
    return pd.DataFrame(new_columns)

# Apply the function
pre_metadata_df_group_region_ref_alt = combine_columns(pre_metadata_df_group_region_ref_alt)

In [324]:
pre_metadata_df_group_region_ref_alt.rename(columns=lambda x: x.replace('region_', '') if 'region_' in x else x , inplace=True)
# Converting float columns to integers
pre_metadata_df_group_region_ref_alt['start'] = pre_metadata_df_group_region_ref_alt['start'].astype(int)
pre_metadata_df_group_region_ref_alt['end'] = pre_metadata_df_group_region_ref_alt['end'].astype(int)

#### Compute variant position
- is the variant position 1-based - yes
- start 0-based: chr10	103716151	103716421 (https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A103716151%2D103716155&hgsid=2386973973_bZuxFLEG8XaC4W7DQS5WrAWSFQGO)

In [325]:
pre_metadata_df_group_region_vcf = pre_metadata_df_group_region_ref_alt.copy()
print(pre_metadata_df_group_region_vcf.shape[0])

44


In [326]:
pre_metadata_df_group_region_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]].head()


,name,sequence,var_pos,chr,start,end,variant_pos,allele
0,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,CAGCTGCCCATGCTGGGACTGTGATTTTTTGTATCCTGAGTTACAC...,NaN,chr1,154813248,154813518,NA,ref
1,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGAGCCCTTCTGGGGGCCCTGGCCACTGGCCACTGGTGGAAGTGTT...,NaN,chr1,154839744,154840014,NA,ref
2,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGAAAGGCACTGGAAATTGTACTTACTCCATTTGGTTTGTCTATT...,NaN,chr1,154840018,154840288,NA,ref
3,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,TTTGCAAAGGTATGGTTGGTGGATGGAGAAAAAGCGGCGTGTGAGA...,NaN,chr1,154840331,154840601,NA,ref
4,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,ACAAATTGCTAACTGAGTGTAGAATAACAGGGCCCCATGGGGTAAG...,NaN,chr1,154860467,154860737,NA,ref


In [ ]:
# def get_var_pos(row, var_pos_column, allele_column, seq_start_column, variant_1_based=True, start_0_based=True, sequence_length=270):
#     """
#     Computes the variant pos (0-based coordinate of variant in the string)
#     for the alternative sequences with different cased of the data
#     (variant_1_based and start_0_based is the default)

#     If variant is 0 based set variant_1_based to false (same goes for start)
#     """
#     variant_position = 'NA'
#     if hf.is_alternative(row[allele_column]):
#         variant_0_based = row[var_pos_column] - 1 if variant_1_based else row[var_pos_column]
#         seq_start_0_based = row[seq_start_column] if start_0_based else row[seq_start_column] - 1
#         variant_position = variant_0_based - seq_start_0_based
#         if row[col_strand] == '-':
#             variant_position = sequence_length - (variant_position + 1) # 0-based variant position
#     return variant_position

import re
def find_indel_pattern(row, ref_column, alt_column):
    """Check if in ref or alt is more than 1 subsequent nucleotide indicating an indel
    Special case: I want it to check for the notation of muliallelic variants as well (A,T) if any of these is an indel"""
    if not hf.is_alternative(row[col_allele]):
        return False
    pattern = r'(^[ACGT]{2,})|(,[ACGT]{2,})'
    # Check if the text matches the pattern
    ref_indel = bool(re.match(pattern, row[ref_column]))
    alt_indel = bool(re.match(pattern, row[alt_column]))
    return ref_indel or alt_indel

In [328]:
# pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)]

In [329]:
pre_metadata_df_group_region_vcf.columns
# pre_metadata_df_group_region_vcf.head()

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'REF', 'chr',
       'start', 'end', 'strand', 'ALT', 'var_pos', 'vcf_REF', 'vcf_ALT'],
      dtype='object')

In [330]:
pre_metadata_df_group_region_vcf[col_variant_pos] = pre_metadata_df_group_region_vcf.apply(lambda row: get_var_pos(row, var_pos_column='var_pos', allele_column=col_allele, seq_start_column=col_start, variant_1_based=True, start_0_based=True), axis=1)

pre_metadata_df_group_region_vcf
pre_metadata_df_group_region_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]

# add variant_class: SNV or indel
pre_metadata_df_group_region_vcf["is_indel"] = pre_metadata_df_group_region_vcf.apply(lambda row: find_indel_pattern(row, "vcf_REF", "vcf_ALT"), axis=1)
pre_metadata_df_group_region_vcf[col_variant_class] = pre_metadata_df_group_region_vcf['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')
pre_metadata_df_group_region_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele, 'vcf_REF', 'vcf_ALT', col_variant_class]]
pre_metadata_df_group_region_vcf[col_variant_class].value_counts() # 2 indels

variant_class
SNV    44
Name: count, dtype: int64

In [331]:
# check if the variant position is int:
pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][col_variant_pos].to_list()[:3]

[195.0, 179.0, 135.0]

In [332]:
pre_metadata_df_group_region_vcf['real_ALT'] = pre_metadata_df_group_region_vcf.apply(lambda row: get_variant_alternative(row, col_sequence=col_sequence, col_variant_pos=col_variant_pos, col_allele=col_allele), axis=1)
pre_metadata_df_group_region_vcf[pre_metadata_df_group_region_vcf[col_variant_class] == 'SNV'][[col_chr, 'var_pos', col_name, col_sequence, col_chr, col_start, col_end, col_strand, 'variant_pos', col_allele,  col_variant_class, 'vcf_REF', 'vcf_ALT', 'real_ALT']].head()


,chr,var_pos,name,sequence,chr,start,end,strand,variant_pos,allele,variant_class,vcf_REF,vcf_ALT,real_ALT
0,chr1,NaN,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,CAGCTGCCCATGCTGGGACTGTGATTTTTTGTATCCTGAGTTACAC...,chr1,154813248,154813518,+,NA,ref,SNV,NaN,NaN,NA
1,chr1,NaN,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGAGCCCTTCTGGGGGCCCTGGCCACTGGCCACTGGTGGAAGTGTT...,chr1,154839744,154840014,+,NA,ref,SNV,NaN,NaN,NA
2,chr1,NaN,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGAAAGGCACTGGAAATTGTACTTACTCCATTTGGTTTGTCTATT...,chr1,154840018,154840288,+,NA,ref,SNV,NaN,NaN,NA
3,chr1,NaN,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,TTTGCAAAGGTATGGTTGGTGGATGGAGAAAAAGCGGCGTGTGAGA...,chr1,154840331,154840601,+,NA,ref,SNV,NaN,NaN,NA
4,chr1,NaN,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,ACAAATTGCTAACTGAGTGTAGAATAACAGGGCCCCATGGGGTAAG...,chr1,154860467,154860737,+,NA,ref,SNV,NaN,NaN,NA


#### Add SPDI

In [333]:
# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

In [334]:
# add SPDI for alt
pre_metadata_df_group_region_vcf['SPDI'] = pre_metadata_df_group_region_vcf.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{int(row['var_pos'])}-{row['vcf_REF']}-{row['real_ALT']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_region_df_group.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict

# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict


pre_metadata_df_group_region_vcf = pre_metadata_df_group_region_vcf.apply(lambda row: add_spdi_values_2_reference(row, ref_alt_dict=ref_alt_dict, alt_spdi_dict=alt_spdi_dict, alt_variant_pos_dict=alt_variant_pos_dict, alt_variant_class_dict=alt_variant_class_dict), axis=1)
pre_metadata_df_group_region_vcf = pre_metadata_df_group_region_vcf.apply(make_column_arrays, axis = 1)

In [335]:
pre_metadata_df_group_region_vcf_final = pre_metadata_df_group_region_vcf[interesting_columns].copy()
pre_metadata_df_group_region_vcf_final.head()

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,CAGCTGCCCATGCTGGGACTGTGATTTTTTGTATCCTGAGTTACAC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154813248,154813518,+,[SNV],[195],[NC_000001.11:154813443:G:A],[ref],
1,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGAGCCCTTCTGGGGGCCCTGGCCACTGGCCACTGGTGGAAGTGTT...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154839744,154840014,+,[SNV],[179],[NC_000001.11:154839923:G:C],[ref],
2,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGAAAGGCACTGGAAATTGTACTTACTCCATTTGGTTTGTCTATT...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154840018,154840288,+,[SNV],[135],[NC_000001.11:154840153:T:C],[ref],
3,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,TTTGCAAAGGTATGGTTGGTGGATGGAGAAAAAGCGGCGTGTGAGA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154840331,154840601,+,[SNV],[135],[NC_000001.11:154840466:G:A],[ref],
4,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,ACAAATTGCTAACTGAGTGTAGAATAACAGGGCCCCATGGGGTAAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154860467,154860737,+,"[SNV, SNV]","[115, 123]","[NC_000001.11:154860582:T:C, NC_000001.11:1548...","[ref, ref]",


In [336]:
# concatenate element and variant results
combined_metadata_final = pd.concat([pre_metadata_df_group_region_vcf_final, pre_metadata_df_group_element_region_final], ignore_index=True)

In [337]:
combined_metadata_final.head()

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,CAGCTGCCCATGCTGGGACTGTGATTTTTTGTATCCTGAGTTACAC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154813248,154813518,+,[SNV],[195],[NC_000001.11:154813443:G:A],[ref],
1,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGAGCCCTTCTGGGGGCCCTGGCCACTGGCCACTGGTGGAAGTGTT...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154839744,154840014,+,[SNV],[179],[NC_000001.11:154839923:G:C],[ref],
2,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGAAAGGCACTGGAAATTGTACTTACTCCATTTGGTTTGTCTATT...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154840018,154840288,+,[SNV],[135],[NC_000001.11:154840153:T:C],[ref],
3,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,TTTGCAAAGGTATGGTTGGTGGATGGAGAAAAAGCGGCGTGTGAGA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154840331,154840601,+,[SNV],[135],[NC_000001.11:154840466:G:A],[ref],
4,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,ACAAATTGCTAACTGAGTGTAGAATAACAGGGCCCCATGGGGTAAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,154860467,154860737,+,"[SNV, SNV]","[115, 123]","[NC_000001.11:154860582:T:C, NC_000001.11:1548...","[ref, ref]",


In [338]:
print('expected number: ', expected_number)
print('number of rows in the metadata file: ', combined_metadata_final.shape[0])

expected number:  45
number of rows in the metadata file:  45


In [339]:
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
combined_metadata_final[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0

## GC_Kircher (n: 203)

In [199]:
group_name = 'GC_Kircher'

pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
expected_number = pre_metadata_df_group.shape[0]
variant_region_df_group = variant_region_map.loc[variant_region_map['tmp_label'] == group_name].copy()
vcf_df_group = vcf_df.loc[vcf_df['ID'].str.contains(group_name)].copy()

In [200]:
# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])

# if variant related or element
variant_related_list = set(variant_region_df_group['REF'].to_list()).union(set(variant_region_df_group['ALT'].to_list()))
pre_metadata_df_group[col_category] = pre_metadata_df_group[col_name].apply(lambda name: 'variant' if is_variant_related(name, variant_related_list) else 'element')


pre_metadata_df_group[col_class] = pre_metadata_df_group[col_name].apply(lambda name: 'variant positive control' if is_variant_related(name, variant_related_list) else 'element active control')
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add allele
alternative_related_list = set(variant_region_df_group['ALT'].to_list())
reference_related_list = set(variant_region_df_group['REF'].to_list())
pre_metadata_df_group[col_allele] = pre_metadata_df_group[col_name].apply(lambda name: 'alt' if is_alternative_related(name, alternative_related_list) else 'ref' if is_reference_related(name, reference_related_list) else 'NA')

pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

In [201]:
pre_metadata_df_group[col_category].value_counts()

category
variant    203
Name: count, dtype: int64

### Focus on Elements

In [202]:
pre_metadata_df_group_element = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'element']
pre_metadata_df_group_element.name.to_list()


[]

### Focus on Variants

In [203]:
pre_metadata_df_group_variant = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'variant'].copy()
print(pre_metadata_df_group_variant.shape[0])
pre_metadata_df_group_variant[col_name].nunique()

203


203

#### Merge the vcf information to the variant map: rename the variant ids

##### Check vcf file header

In [204]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group, on='ID', how='inner')
variant_region_vcf_group # with simple matching only 2 matches (expected 198)

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand,CHROM,var_pos,vcf_REF,vcf_ALT,QUAL,FILTER,INFO
0,GC_Kircher:NC000001_11_109274794_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274659,109274929,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109274795,C,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
1,GC_Kircher:NC000001_11_109274836_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274659,109274929,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109274837,C,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
2,GC_Kircher:NC000001_11_109274836_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274771,109275041,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109274837,C,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
3,GC_Kircher:NC000001_11_109274840_A_C,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274659,109274929,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109274841,A,C,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
4,GC_Kircher:NC000001_11_109274840_A_C,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274771,109275041,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109274841,A,C,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274993,109275263,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109275172,G,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
192,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109275105,109275375,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109275172,G,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
193,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274993,109275263,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109275180,A,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
194,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109275105,109275375,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,chr1,109275180,A,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...


- Understand the names: 
    - 5tiles: references
    - multiple variants tested in each element: 
    - 109274794 - 109275240 
    - from grep command of one reference line: 100 variants in the reference (+1 per alternative)
    - `!cat {input_design_file} | grep "GC_Kircher" | grep "REF" | head -n 1 | grep -o 'NC000001.11' | wc -l`
    - Idea: generate new ID in region bed => matchable with vcf

In [205]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group, on='ID', how='inner')
variant_region_vcf_group
matchable_names = set(variant_region_vcf_group['ID'].to_list())

all_names = set(variant_region_df_group['ID'].to_list())
all_names - matchable_names #{'GC_Kircher:NC000001_11_109274967_G_A'} (is in the design)

{'GC_Kircher:NC000001_11_109274967_G_A'}

In [206]:
# find the corresponding row in the vcf vcf_df_group
vcf_df_group.loc[vcf_df_group['ID'].str.contains('109274967_G')] # neue id: 'GC_Selvarajan:rs12740374;GC_Kircher:NC000001_11_109274967_G_A;C_positive_heart_CAD:rs12740374'

,CHROM,var_pos,ID,vcf_REF,vcf_ALT,QUAL,FILTER,INFO
3180,chr1,109274968,GC_Selvarajan:rs12740374;GC_Kircher:NC000001_1...,G,"T,A",.,PASS,rsid=rs12740374;Region=rs12740374|STARR-seq-He...


##### Rename the ids: 

In [207]:
def rename_ids_with_tiles(row, col_name):
    """
    Get a similar ID as the vcf id but because of the bigger tiles the id was repetitive
    """
    name = row[col_name]
    if'GC_Kircher' not in name:
        return 'NA'

    return 'GC_Kircher:'+'_'.join(name.split('tile')[-1].split('_')[1:])+('_tile' + name.split('tile')[-1].split('_')[0])

In [208]:
example_string = variant_region_df_group['ID'].to_list()[2]
'_'.join(example_string.split('tile')[-1].split('_')[1:])+('_tile' + example_string.split('tile')[-1].split('_')[0])

# expand: split _tile, expand = true to have a matchable ID with the vcf
variant_region_df_group['new_ID'] = variant_region_df_group.apply(lambda row: rename_ids_with_tiles(row, 'ALT'), axis=1)

variant_region_df_group['ID'] = variant_region_df_group['new_ID'].apply(lambda id: id.split('_tile')[0])

Preprocess the vcf id column to make it matchable

In [209]:
# Function to split the IDs and create new rows while conserving all columns
def split_ids(row, id_col):
    ids = row[id_col].split(';')
    new_rows = []
    for id in ids:
        new_row = row.copy()
        new_row[id_col] = id
        new_rows.append(new_row)
    return pd.DataFrame(new_rows)

# Apply the function to each row and concatenate the results
vcf_df_group_split = pd.concat(vcf_df_group.apply(lambda row: split_ids(row, id_col='ID'), axis=1).values)

# Reset the index
vcf_df_group_split.reset_index(drop=True, inplace=True)

print(vcf_df_group_split.shape[0]) # 102 (100 before but one header had multiple columns)

102


In [210]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group_split, on='ID', how='inner')
variant_region_vcf_group.head() # 198

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand,new_ID,CHROM,var_pos,vcf_REF,vcf_ALT,QUAL,FILTER,INFO
0,GC_Kircher:NC000001_11_109274794_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274659,109274929,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,GC_Kircher:NC000001_11_109274794_C_T_tile1-5,chr1,109274795,C,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
1,GC_Kircher:NC000001_11_109274836_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274659,109274929,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,GC_Kircher:NC000001_11_109274836_C_T_tile1-5,chr1,109274837,C,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
2,GC_Kircher:NC000001_11_109274836_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274771,109275041,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,GC_Kircher:NC000001_11_109274836_C_T_tile2-5,chr1,109274837,C,T,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
3,GC_Kircher:NC000001_11_109274840_A_C,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274659,109274929,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,GC_Kircher:NC000001_11_109274840_A_C_tile1-5,chr1,109274841,A,C,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...
4,GC_Kircher:NC000001_11_109274840_A_C,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274771,109275041,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+,GC_Kircher:NC000001_11_109274840_A_C_tile2-5,chr1,109274841,A,C,.,PASS,Region=NC000001.11|109274794|C|T|KircherContro...


In [211]:
variant_region_vcf_group.columns

Index(['ID', 'Region', 'REF', 'ALT', 'tmp_label', 'region_chr', 'region_start',
       'region_end', 'region_name', 'region_score', 'region_strand', 'new_ID',
       'CHROM', 'var_pos', 'vcf_REF', 'vcf_ALT', 'QUAL', 'FILTER', 'INFO'],
      dtype='object')

### Add information to metadata file

In [212]:
only_reference_sequences = variant_region_vcf_group[['REF', 'region_chr', 'region_start', 'region_end', 'region_strand']].drop_duplicates(subset=['REF', 'region_chr', 'region_start', 'region_end', 'region_strand'])
pre_metadata_df_group_region_ref = pre_metadata_df_group_variant.merge(only_reference_sequences, left_on=col_name, right_on='REF', how='left')
print(pre_metadata_df_group_region_ref.shape[0])
print(pre_metadata_df_group_region_ref[col_name].nunique())
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref.merge(variant_region_vcf_group[['ALT', 'region_chr', 'region_start', 'region_end', 'region_strand', 'var_pos', 'vcf_REF', 'vcf_ALT']], left_on=col_name, right_on='ALT', how='left')
print(pre_metadata_df_group_region_ref_alt.shape[0])
print(pre_metadata_df_group_region_ref_alt[col_name].nunique())

203
203
203
203


In [213]:
pre_metadata_df_group_region_ref_alt.columns

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'REF',
       'region_chr_x', 'region_start_x', 'region_end_x', 'region_strand_x',
       'ALT', 'region_chr_y', 'region_start_y', 'region_end_y',
       'region_strand_y', 'var_pos', 'vcf_REF', 'vcf_ALT'],
      dtype='object')

In [214]:
pre_metadata_df_group_region_ref_alt.head()

,name,sequence,tmp_label,category,class,source,ref,allele,variant_class,variant_pos,...,region_end_x,region_strand_x,ALT,region_chr_y,region_start_y,region_end_y,region_strand_y,var_pos,vcf_REF,vcf_ALT
0,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AAAGCCCTGTCCGGTGAGGGGGCAGAAGGACTCAGCGCCCCTGGAC...,GC_Kircher,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,109274929.0,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGCCCCTCTCCTTTTCCTGGACTCTGGCCGTGCGCGGCAGCCCAGG...,GC_Kircher,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,109275041.0,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,CAGTCATGTGTTAAGTTGCGCTTCTTTGCTGTGATGTGGGTGGGGG...,GC_Kircher,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,109275152.0,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,GC_Kircher,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,109275263.0,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,GC_Kircher,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,109275375.0,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Combine columns with _x and _y

In [215]:
# Combine columns
def combine_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith("_x"):
            base_name = col[:-2]  # Remove "_x"
            corresponding_y = base_name + "_y"
            # Combine _x and _y columns
            if corresponding_y in df.columns:
                new_columns[base_name] = df[col].fillna(df[corresponding_y])
            else:
                new_columns[base_name] = df[col]
        elif not col.endswith("_y"):  # Add columns that aren't paired with _x/_y
            new_columns[col] = df[col]
    return pd.DataFrame(new_columns)

# Apply the function
pre_metadata_df_group_region_ref_alt = combine_columns(pre_metadata_df_group_region_ref_alt)

In [216]:
pre_metadata_df_group_region_ref_alt.rename(columns=lambda x: x.replace('region_', '') if 'region_' in x else x , inplace=True)
# Converting float columns to integers
pre_metadata_df_group_region_ref_alt['start'] = pre_metadata_df_group_region_ref_alt['start'].astype(int)
pre_metadata_df_group_region_ref_alt['end'] = pre_metadata_df_group_region_ref_alt['end'].astype(int)

In [217]:
pre_metadata_df_group_region_ref_alt.columns

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'REF', 'chr',
       'start', 'end', 'strand', 'ALT', 'var_pos', 'vcf_REF', 'vcf_ALT'],
      dtype='object')

#### Compute variant position
- is the variant position 1-based - yes
- start 0-based: chr1	109274993	109275263 (https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A109274993%2D109274999&hgsid=2386973973_bZuxFLEG8XaC4W7DQS5WrAWSFQGO)

In [218]:
pre_metadata_df_group_region_vcf = pre_metadata_df_group_region_ref_alt.copy()
print(pre_metadata_df_group_region_vcf.shape[0])

203


In [219]:
pre_metadata_df_group_region_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]


,name,sequence,var_pos,chr,start,end,variant_pos,allele
0,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AAAGCCCTGTCCGGTGAGGGGGCAGAAGGACTCAGCGCCCCTGGAC...,NaN,chr1,109274659,109274929,NA,ref
1,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGCCCCTCTCCTTTTCCTGGACTCTGGCCGTGCGCGGCAGCCCAGG...,NaN,chr1,109274771,109275041,NA,ref
2,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,CAGTCATGTGTTAAGTTGCGCTTCTTTGCTGTGATGTGGGTGGGGG...,NaN,chr1,109274882,109275152,NA,ref
3,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,NaN,chr1,109274993,109275263,NA,ref
4,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,NaN,chr1,109275105,109275375,NA,ref
...,...,...,...,...,...,...,...,...
198,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,109275172.0,chr1,109274993,109275263,NA,alt
199,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,109275172.0,chr1,109275105,109275375,NA,alt
200,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,109275180.0,chr1,109274993,109275263,NA,alt
201,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,109275180.0,chr1,109275105,109275375,NA,alt


In [220]:
# def get_var_pos(row, var_pos_column, allele_column, seq_start_column, variant_1_based=True, start_0_based=True, sequence_length=270):
#     """
#     Computes the variant pos (0-based coordinate of variant in the string)
#     for the alternative sequences with different cased of the data
#     (variant_1_based and start_0_based is the default)

#     If variant is 0 based set variant_1_based to false (same goes for start)
#     """
#     variant_position = 'NA'
#     if hf.is_alternative(row[allele_column]):
#         variant_0_based = row[var_pos_column] - 1 if variant_1_based else row[var_pos_column]
#         seq_start_0_based = row[seq_start_column] if start_0_based else row[seq_start_column] - 1
#         variant_position = variant_0_based - seq_start_0_based
#         if row[col_strand] == '-':
#             variant_position = sequence_length - (variant_position + 1) # 0-based variant position
#     return variant_position

import re
def find_indel_pattern(row, ref_column, alt_column):
    """Check if in ref or alt is more than 1 subsequent nucleotide indicating an indel
    Special case: I want it to check for the notation of muliallelic variants as well (A,T) if any of these is an indel"""
    if not hf.is_alternative(row[col_allele]):
        return False
    pattern = r'(^[ACGT]{2,})|(,[ACGT]{2,})'
    # Check if the text matches the pattern
    ref_indel = bool(re.match(pattern, row[ref_column]))
    alt_indel = bool(re.match(pattern, row[alt_column]))
    return ref_indel or alt_indel


In [221]:
pre_metadata_df_group_region_vcf.columns
pre_metadata_df_group_region_vcf['vcf_ALT']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
198      T
199      T
200      T
201      T
202      A
Name: vcf_ALT, Length: 203, dtype: object

In [222]:
pre_metadata_df_group_region_vcf[col_variant_pos] = pre_metadata_df_group_region_vcf.apply(lambda row: get_var_pos(row, var_pos_column='var_pos', allele_column=col_allele, seq_start_column=col_start, variant_1_based=True, start_0_based=True), axis=1)

pre_metadata_df_group_region_vcf
pre_metadata_df_group_region_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]

# add variant_class: SNV or indel
pre_metadata_df_group_region_vcf["is_indel"] = pre_metadata_df_group_region_vcf.apply(lambda row: find_indel_pattern(row, "vcf_REF", "vcf_ALT"), axis=1)
pre_metadata_df_group_region_vcf[col_variant_class] = pre_metadata_df_group_region_vcf['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')
pre_metadata_df_group_region_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele, 'vcf_REF', 'vcf_ALT', col_variant_class]]
pre_metadata_df_group_region_vcf[col_variant_class].value_counts() # 2 indels
# pre_metadata_df_group_region_vcf[pre_metadata_df_group_region_vcf[col_variant_class] == 'indel'][[col_chr, 'var_pos', 'vcf_REF', 'vcf_ALT', col_name, col_sequence, col_chr, col_start, col_end, 'variant_pos', col_allele,  col_variant_class]]
# pre_metadata_df_group_region_vcf[pre_metadata_df_group_region_vcf[col_variant_class] == 'indel'][col_sequence].to_list()[0][172] # G
# pre_metadata_df_group_region_vcf[pre_metadata_df_group_region_vcf[col_variant_class] == 'indel'][col_sequence].to_list()[1][172] # T

variant_class
SNV    203
Name: count, dtype: int64

In [223]:
# check if the variant position is int:
pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][col_variant_pos].to_list()[:3]


[135.0, 177.0, 65.0]

In [226]:
# pre_metadata_df_group_region_vcf['real_ALT'] = pre_metadata_df_group_region_vcf.apply(lambda row: get_variant_alternative(row, col_sequence=col_sequence, col_variant_pos=col_variant_pos, col_allele=col_allele), axis=1)
pre_metadata_df_group_region_vcf[pre_metadata_df_group_region_vcf[col_variant_class] == 'SNV'][[col_chr, 'var_pos', 'vcf_REF', 'vcf_ALT', col_name, col_sequence, col_chr, col_start, col_end, col_strand, 'variant_pos', col_allele,  col_variant_class]]


,chr,var_pos,vcf_REF,vcf_ALT,name,sequence,chr,start,end,strand,variant_pos,allele,variant_class
0,chr1,NaN,NaN,NaN,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AAAGCCCTGTCCGGTGAGGGGGCAGAAGGACTCAGCGCCCCTGGAC...,chr1,109274659,109274929,+,NA,ref,SNV
1,chr1,NaN,NaN,NaN,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGCCCCTCTCCTTTTCCTGGACTCTGGCCGTGCGCGGCAGCCCAGG...,chr1,109274771,109275041,+,NA,ref,SNV
2,chr1,NaN,NaN,NaN,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,CAGTCATGTGTTAAGTTGCGCTTCTTTGCTGTGATGTGGGTGGGGG...,chr1,109274882,109275152,+,NA,ref,SNV
3,chr1,NaN,NaN,NaN,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,chr1,109274993,109275263,+,NA,ref,SNV
4,chr1,NaN,NaN,NaN,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,chr1,109275105,109275375,+,NA,ref,SNV
...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,chr1,109275172.0,G,T,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,chr1,109274993,109275263,+,178.0,alt,SNV
199,chr1,109275172.0,G,T,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,chr1,109275105,109275375,+,66.0,alt,SNV
200,chr1,109275180.0,A,T,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,chr1,109274993,109275263,+,186.0,alt,SNV
201,chr1,109275180.0,A,T,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,chr1,109275105,109275375,+,74.0,alt,SNV


shorten the name

In [227]:
def extract_reference_from_kircher_name(name):
    """Gets kircher header
    1. if GC_Kircher:ALT_:
        remove the last underscore

    remove the first two underscores
    remove all occurences of KircherControls
    and return as reference name
    """
    if name.startswith("GC_Kircher:ALT_"):
        name = "_".join(name.split("_")[:-5])
    region_name = "_".join(name.split("_")[2:])
    return region_name


def extract_variant_info_from_kircher_name(name):
    if name.startswith("GC_Kircher:ALT_"):
        return "_".join(name.split("_")[-5:])
    return pd.NA


def shorten_kircher_control_name(row, tile_col="tile_info", variant_col="variant_info", name_col="name", chr_col="chr", start_col="start", end_col="end"):
    """
    Remove the region info and only have GC_Kircher:REF_<tile_info> or GC_Kircher:ALT_<tile_info>_variant_info
    """
    name = row[name_col]
    tile_info = row[tile_col]
    ref_alt_info = "_".join(name.split("_")[:2])
    region_info = f'{row[chr_col]}:{row[start_col]}-{row[end_col]}'
    new_name = f'{ref_alt_info}_SORT1_{region_info}_{tile_info}' # only SORT1 regions santiy checked with Max
    if "GC_Kircher:ALT" in ref_alt_info:
        variant_info = row[variant_col]
        return f"{new_name}_{variant_info}"
    return new_name



In [228]:
# gather info
pre_metadata_df_group_region_vcf['reference'] = pre_metadata_df_group_region_vcf[col_name].apply(extract_reference_from_kircher_name)
pre_metadata_df_group_region_vcf['variant_info'] = pre_metadata_df_group_region_vcf[col_name].apply(extract_variant_info_from_kircher_name)
pre_metadata_df_group_region_vcf["tile_info"] = pre_metadata_df_group_region_vcf["reference"].apply(lambda ref: ref.split("_")[-1])

# shorten the name
pre_metadata_df_group_region_vcf["short_name"] = pre_metadata_df_group_region_vcf.apply(lambda row: shorten_kircher_control_name(row, tile_col="tile_info", variant_col="variant_info", name_col="name", chr_col="chr", start_col="start", end_col="end"), axis=1)


Problem: two cases have in the vcf a list T,A
- GC_Kircher:ALT_SORT1_chr1:109274771-109275041_tile2-5_NC000001_11_109274967_G_A
- GC_Kircher:ALT_SORT1_chr1:109274882-109275152_tile3-5_NC000001_11_109274967_G_A

In [232]:
# use the variant info
def get_alt_from_kircher_variant_info(variant_info):
    if isinstance(variant_info, str):
        return variant_info.split("_")[-1]
    return pd.NA

pre_metadata_df_group_region_vcf["variant_info_alt"] = pre_metadata_df_group_region_vcf["variant_info"].apply(get_alt_from_kircher_variant_info)

In [234]:
pre_metadata_df_group_region_vcf.loc[~(pre_metadata_df_group_region_vcf["vcf_ALT"] == pre_metadata_df_group_region_vcf["variant_info_alt"])][["short_name", "vcf_ALT", "variant_info_alt"]]

,short_name,vcf_ALT,variant_info_alt
0,GC_Kircher:REF_SORT1_chr1:109274659-109274929_...,NaN,<NA>
1,GC_Kircher:REF_SORT1_chr1:109274771-109275041_...,NaN,<NA>
2,GC_Kircher:REF_SORT1_chr1:109274882-109275152_...,NaN,<NA>
3,GC_Kircher:REF_SORT1_chr1:109274993-109275263_...,NaN,<NA>
4,GC_Kircher:REF_SORT1_chr1:109275105-109275375_...,NaN,<NA>
80,GC_Kircher:ALT_SORT1_chr1:109274771-109275041_...,"T,A",A
81,GC_Kircher:ALT_SORT1_chr1:109274882-109275152_...,"T,A",A


#### Add SPDI

In [235]:
# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

In [236]:
# add SPDI for alt
pre_metadata_df_group_region_vcf['SPDI'] = pre_metadata_df_group_region_vcf.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{int(row['var_pos'])}-{row['vcf_REF']}-{row['variant_info_alt']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_region_df_group.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict

# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = pre_metadata_df_group_region_vcf.loc[pre_metadata_df_group_region_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict

pre_metadata_df_group_region_vcf = pre_metadata_df_group_region_vcf.apply(lambda row: add_spdi_values_2_reference(row, ref_alt_dict=ref_alt_dict, alt_spdi_dict=alt_spdi_dict, alt_variant_pos_dict=alt_variant_pos_dict, alt_variant_class_dict=alt_variant_class_dict), axis=1)
pre_metadata_df_group_region_vcf = pre_metadata_df_group_region_vcf.apply(make_column_arrays, axis = 1)

In [239]:
pre_metadata_df_group_region_vcf_final = pre_metadata_df_group_region_vcf[interesting_columns].copy()
pre_metadata_df_group_region_vcf_final.head()

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AAAGCCCTGTCCGGTGAGGGGGCAGAAGGACTCAGCGCCCCTGGAC...,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,chr1,109274659,109274929,+,"[SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, ...","[135, 177, 181, 186, 187, 193, 198, 201, 206, ...","[NC_000001.11:109274794:C:T, NC_000001.11:1092...","[ref, ref, ref, ref, ref, ref, ref, ref, ref, ...",
1,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGCCCCTCTCCTTTTCCTGGACTCTGGCCGTGCGCGGCAGCCCAGG...,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,chr1,109274771,109275041,+,"[SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, ...","[65, 69, 74, 75, 81, 86, 89, 94, 98, 113, 114,...","[NC_000001.11:109274836:C:T, NC_000001.11:1092...","[ref, ref, ref, ref, ref, ref, ref, ref, ref, ...",
2,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,CAGTCATGTGTTAAGTTGCGCTTCTTTGCTGTGATGTGGGTGGGGG...,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,chr1,109274882,109275152,+,"[SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, ...","[26, 28, 30, 35, 40, 41, 42, 43, 44, 45, 46, 4...","[NC_000001.11:109274908:T:G, NC_000001.11:1092...","[ref, ref, ref, ref, ref, ref, ref, ref, ref, ...",
3,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,TCTGGGTTCTGGTGTCCACTCACCCACCCCACCCCCCAAAATCAGA...,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,chr1,109274993,109275263,+,"[SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, ...","[26, 27, 28, 29, 32, 33, 34, 43, 45, 46, 47, 4...","[NC_000001.11:109275019:C:T, NC_000001.11:1092...","[ref, ref, ref, ref, ref, ref, ref, ref, ref, ...",
4,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,ATTGGCTCTCTTCTTCAAAGGACCAGGTCCTGTTCCTCTTTCTCCC...,variant,variant positive control,general controls IGVF year 1 design 2023,GRCh38,chr1,109275105,109275375,+,"[SNV, SNV, SNV, SNV]","[62, 66, 74, 135]","[NC_000001.11:109275167:C:T, NC_000001.11:1092...","[ref, ref, ref, ref]",


In [240]:
print('expected number: ', expected_number)
print('number of rows in the metadata file: ', pre_metadata_df_group_region_vcf_final.shape[0])

expected number:  203
number of rows in the metadata file:  203


In [ ]:
breaking before writing

SyntaxError: invalid syntax (2541313106.py, line 1)

In [241]:
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
pre_metadata_df_group_region_vcf_final[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0